In [ ]:
# ============================================================
# AMR密度マップ（x-y平面・AMRグリッドあり）
#
# 表示単位
#   距離：AU
#   質量：M_sun
#   密度：M_sun AU^-3
#
# 仕様
#   1. z=0と交差するAMRブロックのみ使用
#   2. 各ブロックからz=0に最も近い1層だけ抽出
#   3. 全体図とズーム図の両方にAMR境界を表示
#   4. 線形補間＋最近傍補間で穴埋め
#   5. ズーム図はbilinear表示
#   6. ズーム半径：2×10^4～5×10^4 AU
#   7. ズーム色範囲：シンク外密度の5～98パーセンタイル
#   8. 最大・最小密度：シンク外かつズーム領域内で検索
#   9. 密度マップ自体にはシンク内部も表示
#  10. VTKヘッダーのコード時刻をyrへ変換
#  11. ファイル名はタイムステップ順
# ============================================================

import os
import re
from collections import defaultdict

import matplotlib.patches as mpatches
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import LogFormatterSciNotation
from scipy.interpolate import griddata


# ============================================================
# 入出力設定
# ============================================================
vtk_dir = os.path.expanduser(
    "~/athena-project/results/〇〇"
)

output_dir = "./xy_density_maps_with_grid"
os.makedirs(output_dir, exist_ok=True)


# ============================================================
# 単位定義
# ============================================================
# Toyouchi.cppのコード単位
M_UNIT_CGS = 4.0e33   # g
L_UNIT_CGS = 6.7e15   # cm
T_UNIT_CGS = 3.34e10  # s

# 物理定数
AU_CGS = 1.495978707e13
MSUN_CGS = 1.98847e33
YEAR_CGS = 365.25 * 24.0 * 3600.0

# コード単位 → 表示単位
LENGTH_UNIT_AU = L_UNIT_CGS / AU_CGS
MASS_UNIT_MSUN = M_UNIT_CGS / MSUN_CGS
TIME_UNIT_YR = T_UNIT_CGS / YEAR_CGS

# code density → M_sun AU^-3
DENSITY_UNIT = (
    MASS_UNIT_MSUN / LENGTH_UNIT_AU**3
)

print("[INFO] Unit conversion factors:")
print(
    f"  1 code length  = "
    f"{LENGTH_UNIT_AU:.6e} AU"
)
print(
    f"  1 code mass    = "
    f"{MASS_UNIT_MSUN:.6e} M_sun"
)
print(
    f"  1 code time    = "
    f"{TIME_UNIT_YR:.6e} yr"
)
print(
    f"  1 code density = "
    f"{DENSITY_UNIT:.6e} M_sun AU^-3"
)


# ============================================================
# 計算領域
# ============================================================
# 新しい入力ファイル：
# x1, x2, x3 = -224 ～ +224 code length
LBOX_CODE = 448.0

LBOX_AU = (
    LBOX_CODE * LENGTH_UNIT_AU
)

DOMAIN_MIN = -LBOX_AU / 2.0
DOMAIN_MAX = LBOX_AU / 2.0


# ============================================================
# AMR設定
# ============================================================
BASE_NX = 64
MAX_AMR_LEVEL = 5

BASE_DX_CODE = (
    LBOX_CODE / BASE_NX
)

BASE_DX_AU = (
    BASE_DX_CODE * LENGTH_UNIT_AU
)

LEVEL_COLORS = {
    0: "gray",
    1: "blue",
    2: "green",
    3: "black",
    4: "red",
    5: "purple",
}


# ============================================================
# 描画・ズーム設定
# ============================================================
FULL_RESOLUTION = 800
ZOOM_RESOLUTION = 800

# 自動ズーム倍率
ZOOM_RADIUS_FACTOR = 20.0

# 新しい計算領域用
# AU単位で直接指定
ZOOM_RADIUS_MIN_AU = 2.0e4
ZOOM_RADIUS_MAX_AU = 5.0e4

# 全体図の色範囲
FULL_PERCENTILES = (1.0, 99.0)

# ズーム図の色範囲
ZOOM_PERCENTILES = (5.0, 98.0)

FULL_CMAP = "inferno"
ZOOM_CMAP = "turbo"
ZOOM_INTERPOLATION = "bilinear"

dpi = 200
figsize = (19, 8)


# ============================================================
# シンクマスク設定
# ============================================================
# inputファイルのr_sink_auと一致させる
SINK_RADIUS_AU = 1000.0

# シンク境界付近も除外したい場合は
# 1.1～1.2程度に変更する
SINK_MASK_FACTOR = 1.0

SINK_MASK_RADIUS_AU = (
    SINK_MASK_FACTOR * SINK_RADIUS_AU
)

print("[INFO] Domain and mask settings:")
print(
    f"  Domain width = {LBOX_AU:.6e} AU"
)
print(
    f"  Level 0 dx   = {BASE_DX_AU:.6e} AU"
)
print(
    f"  Sink mask    = "
    f"r < {SINK_MASK_RADIUS_AU:.6e} AU"
)
print(
    f"  Zoom range   = "
    f"{ZOOM_RADIUS_MIN_AU:.6e}–"
    f"{ZOOM_RADIUS_MAX_AU:.6e} AU"
)


# ============================================================
# VTKヘッダーからコード時刻を取得
# ============================================================
def read_vtk_time_code(filename):
    with open(filename, "rb") as vtk_file:
        header = vtk_file.read(512).decode(
            "ascii",
            errors="ignore",
        )

    match = re.search(
        r"time\s*=\s*"
        r"([+-]?(?:\d+\.?\d*|\.\d+)"
        r"(?:[eE][+-]?\d+)?)",
        header,
    )

    if match is None:
        raise ValueError(
            f"Could not find time in VTK header: "
            f"{filename}"
        )

    return float(match.group(1))


# ============================================================
# 対数カラースケール
# ============================================================
def calculate_log_limits(
    values,
    percentiles,
    fallback=None,
):
    values = np.asarray(values).ravel()

    positive_values = values[
        np.isfinite(values)
        & (values > 0.0)
    ]

    if len(positive_values) == 0:
        if fallback is not None:
            return fallback

        return 1.0e-17, 1.0e-12

    vmin, vmax = np.percentile(
        positive_values,
        percentiles,
    )

    if not np.isfinite(vmin) or vmin <= 0.0:
        vmin = np.min(positive_values)

    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = np.max(positive_values)

    if vmax <= vmin:
        vmin *= 0.5
        vmax *= 2.0

    # 色範囲が狭すぎる場合は最低1桁確保
    if vmax / vmin < 10.0:
        center = np.sqrt(vmin * vmax)

        vmin = center / np.sqrt(10.0)
        vmax = center * np.sqrt(10.0)

    return vmin, vmax


# ============================================================
# z=0に最も近いセル層を抽出
# ============================================================
def extract_xy_midplane(
    grid,
    density_name,
):
    """
    z=0と交差するAMRブロックから、
    z=0に最も近いセル中心面を1層だけ抽出する。

    Returns
    -------
    points_au
        (x, y, z)座標。単位はAU。
    density
        密度。単位はM_sun AU^-3。
    """
    bounds = grid.bounds

    # bounds = (xmin, xmax, ymin, ymax, zmin, zmax)
    if not (
        bounds[4] <= 0.0 <= bounds[5]
    ):
        return None, None

    points_code = (
        grid.cell_centers().points
    )

    if len(points_code) == 0:
        return None, None

    density_code = np.asarray(
        grid[density_name]
    ).reshape(-1)

    if len(density_code) != len(points_code):
        raise ValueError(
            "Density size does not match "
            "cell-center size."
        )

    z_values = points_code[:, 2]

    z_nearest = z_values[
        np.argmin(np.abs(z_values))
    ]

    tolerance = (
        1.0e-10
        * max(np.max(np.abs(z_values)), 1.0)
    )

    midplane_mask = np.isclose(
        z_values,
        z_nearest,
        rtol=1.0e-10,
        atol=tolerance,
    )

    if not np.any(midplane_mask):
        return None, None

    points_au = (
        points_code[midplane_mask]
        * LENGTH_UNIT_AU
    )

    density = (
        density_code[midplane_mask]
        * DENSITY_UNIT
    )

    return points_au, density


# ============================================================
# AMRブロック情報
# ============================================================
def extract_amr_block(grid):
    """
    z=0と交差するAMRブロックについて、
    x-y境界、AMRレベル、セル幅を取得する。
    """
    bounds = grid.bounds

    if not (
        bounds[4] <= 0.0 <= bounds[5]
    ):
        return None

    points_code = (
        grid.cell_centers().points
    )

    if len(points_code) == 0:
        return None

    x_values = np.unique(
        np.sort(points_code[:, 0])
    )

    if len(x_values) > 1:
        differences = np.diff(x_values)
        differences = differences[
            differences > 0.0
        ]

        if len(differences) > 0:
            dx_code = np.min(differences)
        else:
            dx_code = BASE_DX_CODE
    else:
        dx_code = BASE_DX_CODE

    dx_au = (
        dx_code * LENGTH_UNIT_AU
    )

    level = int(
        np.clip(
            np.round(
                np.log2(
                    BASE_DX_AU / dx_au
                )
            ),
            0,
            MAX_AMR_LEVEL,
        )
    )

    return {
        "bounds": (
            bounds[0] * LENGTH_UNIT_AU,
            bounds[1] * LENGTH_UNIT_AU,
            bounds[2] * LENGTH_UNIT_AU,
            bounds[3] * LENGTH_UNIT_AU,
        ),
        "level": level,
        "dx": dx_au,
    }


# ============================================================
# AMR境界描画
# ============================================================
def draw_amr_blocks(
    ax,
    blocks,
    linewidth=1.0,
):
    """
    境界線の重複を減らすため、
    各ブロックの右辺と上辺だけを描く。
    """
    sorted_blocks = sorted(
        blocks,
        key=lambda block: block["level"],
    )

    for block in sorted_blocks:
        x0, x1, y0, y1 = block["bounds"]
        level = block["level"]

        color = LEVEL_COLORS.get(
            level,
            "white",
        )

        # 右辺
        ax.plot(
            [x1, x1],
            [y0, y1],
            color=color,
            linewidth=linewidth,
            alpha=0.8,
        )

        # 上辺
        ax.plot(
            [x0, x1],
            [y1, y1],
            color=color,
            linewidth=linewidth,
            alpha=0.8,
        )


# ============================================================
# 線形補間＋最近傍補間
# ============================================================
def smooth_interpolation(
    points_2d,
    values,
    X,
    Y,
):
    """
    線形補間を基本とし、
    線形補間できない外縁だけを最近傍補間で埋める。
    """
    linear = griddata(
        points_2d,
        values,
        (X, Y),
        method="linear",
    )

    nearest = griddata(
        points_2d,
        values,
        (X, Y),
        method="nearest",
    )

    result = np.where(
        np.isfinite(linear),
        linear,
        nearest,
    )

    invalid = (
        ~np.isfinite(result)
        | (result <= 0.0)
    )

    result[invalid] = np.nan

    return result


# ============================================================
# VTKファイル整理
# ============================================================
if not os.path.isdir(vtk_dir):
    raise FileNotFoundError(
        f"VTK directory does not exist: {vtk_dir}"
    )

files_by_step = defaultdict(list)

for filename in os.listdir(vtk_dir):
    if not (
        filename.startswith("Toyouchi.block")
        and filename.endswith(".vtk")
    ):
        continue

    match = re.search(
        r"(?:prim\.)?out2\.(\d+)",
        filename,
    )

    if match is not None:
        timestep = int(match.group(1))

        files_by_step[timestep].append(
            os.path.join(
                vtk_dir,
                filename,
            )
        )

steps = sorted(files_by_step)

if not steps:
    raise RuntimeError(
        f"No VTK files found in {vtk_dir}"
    )

print(
    f"[INFO] Found {len(steps)} timesteps"
)
print(
    f"[INFO] Timestep range: "
    f"{steps[0]}–{steps[-1]}"
)


# ============================================================
# 密度変数名検出
# ============================================================
test_file = files_by_step[steps[0]][0]
test_grid = pv.read(test_file)

print(
    f"[INFO] Available arrays: "
    f"{test_grid.array_names}"
)

density_name = next(
    (
        name
        for name in [
            "dens",
            "density",
            "rho",
            "prim_dens",
            "prim_density",
        ]
        if name in test_grid.array_names
    ),
    None,
)

if density_name is None:
    raise RuntimeError(
        "No density array was found."
    )

print(
    f"[INFO] Density variable: "
    f"{density_name}"
)


# ============================================================
# 全体図共通カラースケール
# シンク内部を除外した密度から計算
# ============================================================
sample_indices = np.linspace(
    0,
    len(steps) - 1,
    min(10, len(steps)),
).astype(int)

sample_steps = [
    steps[index]
    for index in sample_indices
]

sample_density = []

for step in sample_steps:
    for filename in files_by_step[step]:
        try:
            grid = pv.read(filename)

            sample_points, density_sample = (
                extract_xy_midplane(
                    grid,
                    density_name,
                )
            )

            if sample_points is None:
                continue

            sample_radius_spherical = (
                np.linalg.norm(
                    sample_points,
                    axis=1,
                )
            )

            sample_outside_sink = (
                sample_radius_spherical
                >= SINK_MASK_RADIUS_AU
            )

            if np.any(sample_outside_sink):
                sample_density.append(
                    density_sample[
                        sample_outside_sink
                    ]
                )

        except Exception as error:
            print(
                f"[WARNING] Could not sample "
                f"{filename}: {error}"
            )

if not sample_density:
    raise RuntimeError(
        "No valid density data outside "
        "the sink were found."
    )

sample_density = np.concatenate(
    sample_density
)

full_vmin, full_vmax = (
    calculate_log_limits(
        sample_density,
        FULL_PERCENTILES,
    )
)

full_norm = LogNorm(
    vmin=full_vmin,
    vmax=full_vmax,
)

print(
    f"[INFO] Global density range "
    f"(outside sink): "
    f"{full_vmin:.3e}–"
    f"{full_vmax:.3e} M_sun AU^-3"
)


# ============================================================
# メイン処理
# ============================================================
for timestep_index, step in enumerate(steps):
    print(
        f"[INFO] Processing timestep "
        f"{step:05d} "
        f"({timestep_index + 1}/"
        f"{len(steps)})"
    )

    point_arrays = []
    density_arrays = []
    amr_blocks = []

    for filename in files_by_step[step]:
        try:
            grid = pv.read(filename)

            points, density = (
                extract_xy_midplane(
                    grid,
                    density_name,
                )
            )

            block = extract_amr_block(grid)

            if points is not None:
                point_arrays.append(points)
                density_arrays.append(density)

            if block is not None:
                amr_blocks.append(block)

        except Exception as error:
            print(
                f"[WARNING] Failed to process "
                f"{filename}: {error}"
            )

    if not point_arrays:
        print(
            f"[WARNING] No x-y midplane data "
            f"at timestep {step:05d}"
        )
        continue

    points = np.vstack(point_arrays)
    density = np.hstack(density_arrays)

    valid = (
        np.all(
            np.isfinite(points),
            axis=1,
        )
        & np.isfinite(density)
        & (density > 0.0)
    )

    points = points[valid]
    density = density[valid]

    if len(density) == 0:
        print(
            f"[WARNING] No valid density data "
            f"at timestep {step:05d}"
        )
        continue

    # ========================================================
    # 時刻
    # ========================================================
    representative_file = (
        files_by_step[step][0]
    )

    time_code = read_vtk_time_code(
        representative_file
    )

    time_yr = (
        time_code * TIME_UNIT_YR
    )

    # ========================================================
    # 平面内半径と球半径
    # ========================================================
    radius_xy = np.hypot(
        points[:, 0],
        points[:, 1],
    )

    # シンク判定には球半径を使用
    radius_spherical = np.linalg.norm(
        points,
        axis=1,
    )

    # ========================================================
    # ズーム半径
    # ========================================================
    nearest_cell_radius = np.min(
        radius_xy
    )

    zoom_radius = np.clip(
        ZOOM_RADIUS_FACTOR
        * nearest_cell_radius,
        ZOOM_RADIUS_MIN_AU,
        ZOOM_RADIUS_MAX_AU,
    )

    # 実データ領域を超えないように制限
    data_half_width = min(
        np.max(np.abs(points[:, 0])),
        np.max(np.abs(points[:, 1])),
    )

    zoom_radius = min(
        zoom_radius,
        data_half_width,
    )

    print(
        f"[INFO] Zoom radius = "
        f"{zoom_radius:.3e} AU"
    )

    # ========================================================
    # 全体図補間
    # ========================================================
    axis_full = np.linspace(
        DOMAIN_MIN,
        DOMAIN_MAX,
        FULL_RESOLUTION,
    )

    X_full, Y_full = np.meshgrid(
        axis_full,
        axis_full,
    )

    density_full = smooth_interpolation(
        points[:, [0, 1]],
        density,
        X_full,
        Y_full,
    )

    # ========================================================
    # ズーム領域
    # ========================================================
    zoom_mask = (
        radius_xy <= zoom_radius
    )

    points_zoom = points[zoom_mask]
    density_zoom_raw = density[zoom_mask]

    radius_spherical_zoom = (
        radius_spherical[zoom_mask]
    )

    if len(points_zoom) <= 10:
        print(
            f"[WARNING] Insufficient zoom data "
            f"at timestep {step:05d}"
        )
        continue

    # ========================================================
    # シンク外部を選ぶ診断用マスク
    # ========================================================
    outside_sink_mask = (
        radius_spherical_zoom
        >= SINK_MASK_RADIUS_AU
    )

    if not np.any(outside_sink_mask):
        print(
            f"[WARNING] No cells outside sink "
            f"at timestep {step:05d}"
        )
        continue

    points_diagnostic = points_zoom[
        outside_sink_mask
    ]

    density_diagnostic = density_zoom_raw[
        outside_sink_mask
    ]

    radius_diagnostic = (
        radius_spherical_zoom[
            outside_sink_mask
        ]
    )

    # ========================================================
    # ズーム補間
    # 描画にはシンク内部も含める
    # ========================================================
    axis_zoom = np.linspace(
        -zoom_radius,
        zoom_radius,
        ZOOM_RESOLUTION,
    )

    X_zoom, Y_zoom = np.meshgrid(
        axis_zoom,
        axis_zoom,
    )

    density_zoom = smooth_interpolation(
        points_zoom[:, [0, 1]],
        density_zoom_raw,
        X_zoom,
        Y_zoom,
    )

    # ========================================================
    # ズーム専用カラースケール
    # シンク外部の密度のみから計算
    # ========================================================
    zoom_vmin, zoom_vmax = (
        calculate_log_limits(
            density_diagnostic,
            ZOOM_PERCENTILES,
            fallback=(
                full_vmin,
                full_vmax,
            ),
        )
    )

    zoom_norm = LogNorm(
        vmin=zoom_vmin,
        vmax=zoom_vmax,
    )

    # ========================================================
    # 最大・最小密度
    # シンク外かつズーム領域内だけを検索
    # ========================================================
    maximum_index = np.argmax(
        density_diagnostic
    )

    minimum_index = np.argmin(
        density_diagnostic
    )

    rho_max = density_diagnostic[
        maximum_index
    ]

    rho_min = density_diagnostic[
        minimum_index
    ]

    r_max = radius_diagnostic[
        maximum_index
    ]

    r_min = radius_diagnostic[
        minimum_index
    ]

    position_max = points_diagnostic[
        maximum_index
    ]

    position_min = points_diagnostic[
        minimum_index
    ]

    print(
        f"[INFO] Density extrema outside sink "
        f"(r >= {SINK_MASK_RADIUS_AU:.3e} AU):"
    )

    print(
        f"  rho_max = {rho_max:.3e} "
        f"M_sun AU^-3 at "
        f"({position_max[0]:.3e}, "
        f"{position_max[1]:.3e}, "
        f"{position_max[2]:.3e}) AU, "
        f"r = {r_max:.3e} AU"
    )

    print(
        f"  rho_min = {rho_min:.3e} "
        f"M_sun AU^-3 at "
        f"({position_min[0]:.3e}, "
        f"{position_min[1]:.3e}, "
        f"{position_min[2]:.3e}) AU, "
        f"r = {r_min:.3e} AU"
    )

    # ========================================================
    # Figure
    # ========================================================
    fig = plt.figure(figsize=figsize)

    gs = GridSpec(
        1,
        5,
        width_ratios=[
            4.0,
            0.25,
            4.0,
            0.25,
            1.6,
        ],
        wspace=0.35,
    )

    # ========================================================
    # 全体図
    # ========================================================
    ax_full = fig.add_subplot(gs[0, 0])
    ax_full.set_aspect("equal")

    image_full = ax_full.pcolormesh(
        X_full,
        Y_full,
        density_full,
        cmap=FULL_CMAP,
        norm=full_norm,
        shading="auto",
        rasterized=True,
    )

    draw_amr_blocks(
        ax_full,
        amr_blocks,
        linewidth=1.0,
    )

    # ズーム範囲
    zoom_rectangle = patches.Rectangle(
        (
            -zoom_radius,
            -zoom_radius,
        ),
        2.0 * zoom_radius,
        2.0 * zoom_radius,
        edgecolor="cyan",
        facecolor="none",
        linestyle="--",
        linewidth=2.0,
    )

    ax_full.add_patch(
        zoom_rectangle
    )

    ax_full.plot(
        0.0,
        0.0,
        "r+",
        markersize=12,
        markeredgewidth=2,
    )

    ax_full.set_xlim(
        DOMAIN_MIN,
        DOMAIN_MAX,
    )

    ax_full.set_ylim(
        DOMAIN_MIN,
        DOMAIN_MAX,
    )

    ax_full.set_xlabel(
        r"$x\ [{\rm AU}]$",
        fontsize=12,
    )

    ax_full.set_ylabel(
        r"$y\ [{\rm AU}]$",
        fontsize=12,
    )

    ax_full.set_title(
        "Full Domain",
        fontsize=12,
        fontweight="bold",
    )

    ax_full.ticklabel_format(
        axis="both",
        style="scientific",
        scilimits=(0, 0),
    )

    ax_full.grid(False)

    # AMR凡例
    legend_elements = []

    for level in range(
        MAX_AMR_LEVEL + 1
    ):
        dx_au = (
            BASE_DX_AU / 2**level
        )

        legend_elements.append(
            mpatches.Patch(
                facecolor="none",
                edgecolor=(
                    LEVEL_COLORS[level]
                ),
                linewidth=1.5,
                label=(
                    f"Level {level}, "
                    f"dx={dx_au:.2e} AU"
                ),
            )
        )

    legend_elements.append(
        mpatches.Patch(
            facecolor="none",
            edgecolor="cyan",
            linestyle="--",
            linewidth=1.5,
            label="Zoom region",
        )
    )

    ax_full.legend(
        handles=legend_elements,
        fontsize=7,
        loc="upper right",
        framealpha=0.75,
    )

    # ========================================================
    # 全体図カラーバー
    # ========================================================
    cax_full = fig.add_subplot(gs[0, 1])

    cbar_full = fig.colorbar(
        image_full,
        cax=cax_full,
        extend="both",
    )

    cbar_full.ax.yaxis.set_major_formatter(
        LogFormatterSciNotation()
    )

    cbar_full.set_label(
        r"$\rho\ "
        r"[M_\odot\,{\rm AU}^{-3}]$"
        "\nGlobal scale",
        fontsize=10,
    )

    # ========================================================
    # ズーム図
    # ========================================================
    ax_zoom = fig.add_subplot(gs[0, 2])
    ax_zoom.set_aspect("equal")

    image_zoom = ax_zoom.imshow(
        density_zoom,
        origin="lower",
        extent=[
            -zoom_radius,
            zoom_radius,
            -zoom_radius,
            zoom_radius,
        ],
        cmap=ZOOM_CMAP,
        norm=zoom_norm,
        interpolation=ZOOM_INTERPOLATION,
        aspect="equal",
        rasterized=True,
    )

    # ズーム範囲と交差するAMRブロックを抽出
    zoom_blocks = []

    for block in amr_blocks:
        x0, x1, y0, y1 = block["bounds"]

        intersects_zoom = (
            x0 <= zoom_radius
            and x1 >= -zoom_radius
            and y0 <= zoom_radius
            and y1 >= -zoom_radius
        )

        if not intersects_zoom:
            continue

        clipped_x0 = max(
            x0,
            -zoom_radius,
        )
        clipped_x1 = min(
            x1,
            zoom_radius,
        )
        clipped_y0 = max(
            y0,
            -zoom_radius,
        )
        clipped_y1 = min(
            y1,
            zoom_radius,
        )

        if (
            clipped_x1 > clipped_x0
            and clipped_y1 > clipped_y0
        ):
            zoom_blocks.append(
                {
                    "bounds": (
                        clipped_x0,
                        clipped_x1,
                        clipped_y0,
                        clipped_y1,
                    ),
                    "level": block["level"],
                }
            )

    draw_amr_blocks(
        ax_zoom,
        zoom_blocks,
        linewidth=1.0,
    )

    ax_zoom.plot(
        0.0,
        0.0,
        "r+",
        markersize=12,
        markeredgewidth=2,
        label="Center",
    )

    ax_zoom.set_xlim(
        -zoom_radius,
        zoom_radius,
    )

    ax_zoom.set_ylim(
        -zoom_radius,
        zoom_radius,
    )

    ax_zoom.set_xlabel(
        r"$x\ [{\rm AU}]$",
        fontsize=12,
    )

    ax_zoom.set_ylabel(
        r"$y\ [{\rm AU}]$",
        fontsize=12,
    )

    ax_zoom.set_title(
        "Zoomed Region\n"
        f"R = {zoom_radius:.3e} AU",
        fontsize=12,
        fontweight="bold",
    )

    ax_zoom.ticklabel_format(
        axis="both",
        style="scientific",
        scilimits=(0, 0),
    )

    ax_zoom.grid(False)

    ax_zoom.legend(
        loc="upper right",
        fontsize=8,
        framealpha=0.7,
    )

    # ========================================================
    # ズーム図カラーバー
    # ========================================================
    cax_zoom = fig.add_subplot(gs[0, 3])

    cbar_zoom = fig.colorbar(
        image_zoom,
        cax=cax_zoom,
        extend="both",
    )

    cbar_zoom.ax.yaxis.set_major_formatter(
        LogFormatterSciNotation()
    )

    cbar_zoom.set_label(
        r"$\rho\ "
        r"[M_\odot\,{\rm AU}^{-3}]$"
        "\n"
        f"Zoom {ZOOM_PERCENTILES[0]:.0f}–"
        f"{ZOOM_PERCENTILES[1]:.0f} percentile"
        "\n(outside sink)",
        fontsize=10,
    )

    # ========================================================
    # 情報欄
    # ========================================================
    ax_info = fig.add_subplot(gs[0, 4])
    ax_info.axis("off")

    information_text = (
        "Time\n"
        f"{time_yr:.6e} yr\n\n"

        "Code time\n"
        f"{time_code:.6e}\n\n"

        "Zoom radius\n"
        f"{zoom_radius:.3e} AU\n\n"

        "Density search region\n"
        f"r >= {SINK_MASK_RADIUS_AU:.3e} AU\n"
        "(outside sink)\n\n"

        "Maximum density\n"
        f"{rho_max:.3e}\n"
        r"$M_\odot\,{\rm AU}^{-3}$"
        "\n"
        f"r = {r_max:.3e} AU\n\n"

        "Minimum density\n"
        f"{rho_min:.3e}\n"
        r"$M_\odot\,{\rm AU}^{-3}$"
        "\n"
        f"r = {r_min:.3e} AU\n\n"

        "Zoom color range\n"
        f"{zoom_vmin:.2e}\n"
        "to\n"
        f"{zoom_vmax:.2e}"
    )

    ax_info.text(
        0.04,
        0.95,
        information_text,
        transform=ax_info.transAxes,
        fontsize=9,
        verticalalignment="top",
        bbox=dict(
            boxstyle="round",
            facecolor="whitesmoke",
            edgecolor="black",
            alpha=0.9,
        ),
    )

    # ========================================================
    # タイトル
    # ========================================================
    fig.suptitle(
        "AMR Density Map: x-y Midplane\n"
        f"t = {time_yr:.6e} yr | "
        f"Variable: {density_name}",
        fontsize=14,
        fontweight="bold",
        y=0.98,
    )

    plt.subplots_adjust(
        top=0.88,
        bottom=0.10,
        left=0.05,
        right=0.97,
    )

    # ========================================================
    # タイムステップ順のファイル名で保存
    # ========================================================
    png = os.path.join(
        output_dir,
        (
            f"density_xy_timestep_"
            f"{step:05d}_"
            f"time_{time_yr:.6e}yr_"
            "amr_blocks.png"
        ),
    )

    fig.savefig(
        png,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)

    print(
        f"[INFO] Saved: "
        f"step={step:05d}, "
        f"time={time_yr:.6e} yr"
    )


print(
    f"[INFO] All AMR density maps "
    f"saved to: {output_dir}"
)

# 密度マップ（x-y平面・AMRグリッド非表示版）
#
# 表示単位
#   距離：AU
#   質量：M_sun
#   密度：M_sun AU^-3
#
# 仕様
#   1. AMRグリッド境界は表示しない
#   2. z=0と交差するブロックのみ使用
#   3. 各ブロックからz=0に最も近い1層だけを抽出
#   4. 線形補間＋最近傍補間による穴埋め
#   5. ズーム図はbilinear補間で滑らかに表示
#   6. 全体図とズーム図で独立した対数カラースケールを使用
# ============================================================

# x-y平面密度マップ（AMRグリッドなし・時刻yr表示）

import os
import re
from collections import defaultdict

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import LogFormatterSciNotation
from scipy.interpolate import griddata


# ============================================================
# 設定
# ============================================================
vtk_dir = os.path.expanduser(
    "~/athena-project/results/〇〇"
)
output_dir = "./xy_density_maps_no_grid"
os.makedirs(output_dir, exist_ok=True)

M_UNIT_CGS = 4.0e33
L_UNIT_CGS = 6.7e15
T_UNIT_CGS = 3.34e10

AU_CGS = 1.495978707e13
MSUN_CGS = 1.98847e33
YEAR_CGS = 365.25 * 24.0 * 3600.0

LENGTH_UNIT_AU = L_UNIT_CGS / AU_CGS
MASS_UNIT_MSUN = M_UNIT_CGS / MSUN_CGS
DENSITY_UNIT = MASS_UNIT_MSUN / LENGTH_UNIT_AU**3
TIME_UNIT_YR = T_UNIT_CGS / YEAR_CGS

LBOX_CODE = 448.0
LBOX_AU = LBOX_CODE * LENGTH_UNIT_AU
DOMAIN_MIN = -LBOX_AU / 2.0
DOMAIN_MAX = LBOX_AU / 2.0

FULL_RESOLUTION = 800
ZOOM_RESOLUTION = 800
ZOOM_RADIUS_FACTOR = 20.0
ZOOM_RADIUS_MIN_AU = 2.0e4
ZOOM_RADIUS_MAX_AU = 5.0e4

FULL_PERCENTILES = (1.0, 99.0)
ZOOM_PERCENTILES = (5.0, 98.0)

FULL_CMAP = "inferno"
ZOOM_CMAP = "turbo"
ZOOM_INTERPOLATION = "bilinear"
# ============================================================
# シンク領域のマスク設定
# inputファイルのr_sink_auと一致させる
# ============================================================
SINK_RADIUS_AU = 1000.0

# 数値誤差やシンク境界直近も除外したい場合は
# 1.1や1.2に変更する
SINK_MASK_FACTOR = 1.0

SINK_MASK_RADIUS_AU = (
    SINK_MASK_FACTOR * SINK_RADIUS_AU
)


dpi = 200
figsize = (19, 8)


# ============================================================
# VTKヘッダーからコード時刻を読み取る
# ============================================================
def read_vtk_time_code(filename):
    with open(filename, "rb") as vtk_file:
        header = vtk_file.read(512).decode(
            "ascii",
            errors="ignore",
        )

    match = re.search(
        r"time\s*=\s*"
        r"([+-]?(?:\d+\.?\d*|\.\d+)"
        r"(?:[eE][+-]?\d+)?)",
        header,
    )

    if match is None:
        raise ValueError(
            f"Could not find time in VTK header: {filename}"
        )

    return float(match.group(1))


def read_vtk_time_yr(filename):
    return read_vtk_time_code(filename) * TIME_UNIT_YR


# ============================================================
# カラースケール
# ============================================================
def log_limits(values, percentiles, fallback=None):
    values = np.asarray(values).ravel()
    values = values[
        np.isfinite(values) & (values > 0.0)
    ]

    if len(values) == 0:
        if fallback is not None:
            return fallback
        return 1.0e-17, 1.0e-12

    vmin, vmax = np.percentile(
        values,
        percentiles,
    )

    if not np.isfinite(vmin) or vmin <= 0.0:
        vmin = np.min(values)
    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = np.max(values)

    if vmax <= vmin:
        vmin *= 0.5
        vmax *= 2.0

    if vmax / vmin < 10.0:
        center = np.sqrt(vmin * vmax)
        vmin = center / np.sqrt(10.0)
        vmax = center * np.sqrt(10.0)

    return vmin, vmax


# ============================================================
# z=0に最も近い1層を抽出
# ============================================================
def extract_xy_midplane(grid, density_name):
    bounds = grid.bounds

    if not (bounds[4] <= 0.0 <= bounds[5]):
        return None, None

    points_code = grid.cell_centers().points
    density_code = np.asarray(
        grid[density_name]
    ).reshape(-1)

    if len(points_code) == 0:
        return None, None

    if len(points_code) != len(density_code):
        raise ValueError(
            "Density size does not match cell-center size."
        )

    z_coordinates = points_code[:, 2]
    z_nearest = z_coordinates[
        np.argmin(np.abs(z_coordinates))
    ]

    tolerance = (
        1.0e-10
        * max(np.max(np.abs(z_coordinates)), 1.0)
    )

    mask = np.isclose(
        z_coordinates,
        z_nearest,
        rtol=1.0e-10,
        atol=tolerance,
    )

    return (
        points_code[mask] * LENGTH_UNIT_AU,
        density_code[mask] * DENSITY_UNIT,
    )


# ============================================================
# 線形補間＋外側だけ最近傍補間
# ============================================================
def smooth_interpolation(points, values, X, Y):
    linear = griddata(
        points,
        values,
        (X, Y),
        method="linear",
    )
    nearest = griddata(
        points,
        values,
        (X, Y),
        method="nearest",
    )

    result = np.where(
        np.isfinite(linear),
        linear,
        nearest,
    )
    result[
        ~np.isfinite(result) | (result <= 0.0)
    ] = np.nan

    return result


# ============================================================
# ファイル整理
# ============================================================
if not os.path.isdir(vtk_dir):
    raise FileNotFoundError(vtk_dir)

files_by_step = defaultdict(list)

for name in os.listdir(vtk_dir):
    if not (
        name.startswith("Toyouchi.block")
        and name.endswith(".vtk")
    ):
        continue

    match = re.search(
        r"(?:prim\.)?out2\.(\d+)",
        name,
    )

    if match:
        step = int(match.group(1))
        files_by_step[step].append(
            os.path.join(vtk_dir, name)
        )

steps = sorted(files_by_step)

if not steps:
    raise RuntimeError("No VTK files found.")

test_grid = pv.read(files_by_step[steps[0]][0])

density_name = next(
    (
        name
        for name in [
            "dens",
            "density",
            "rho",
            "prim_dens",
            "prim_density",
        ]
        if name in test_grid.array_names
    ),
    None,
)

if density_name is None:
    raise RuntimeError("No density array found.")


# ============================================================
# 全体図の共通カラースケール
# ============================================================
sample_steps = [
    steps[index]
    for index in np.linspace(
        0,
        len(steps) - 1,
        min(10, len(steps)),
    ).astype(int)
]

sample_density = []

for step in sample_steps:
    for filename in files_by_step[step]:
        try:
            grid = pv.read(filename)
            _, density = extract_xy_midplane(
                grid,
                density_name,
            )
            if density is not None:
                sample_density.append(density)
        except Exception as error:
            print(f"[WARNING] {filename}: {error}")

if not sample_density:
    raise RuntimeError("No x-y midplane data found.")

full_vmin, full_vmax = log_limits(
    np.concatenate(sample_density),
    FULL_PERCENTILES,
)
full_norm = LogNorm(
    vmin=full_vmin,
    vmax=full_vmax,
)


# ============================================================
# メイン処理
# ============================================================
for index, step in enumerate(steps):
    points_list = []
    density_list = []

    for filename in files_by_step[step]:
        try:
            grid = pv.read(filename)
            points, density = extract_xy_midplane(
                grid,
                density_name,
            )

            if points is not None:
                points_list.append(points)
                density_list.append(density)

        except Exception as error:
            print(f"[WARNING] {filename}: {error}")

    if not points_list:
        continue

    points = np.vstack(points_list)
    density = np.hstack(density_list)

    valid = (
        np.all(np.isfinite(points), axis=1)
        & np.isfinite(density)
        & (density > 0.0)
    )
    points = points[valid]
    density = density[valid]

    if len(density) == 0:
        continue

    time_code = read_vtk_time_code(
        files_by_step[step][0]
    )
    time_yr = time_code * TIME_UNIT_YR

    # ========================================================
    # 球半径とx-y平面内半径
    # ========================================================
    # x-y平面内の円筒半径
    radius_xy = np.hypot(
        points[:, 0],
        points[:, 1],
    )

    # シンクの判定にはシミュレーション本体と同じ球半径を使う
    radius_spherical = np.linalg.norm(
        points,
        axis=1,
    )

    # ========================================================
    # ズーム半径
    # ========================================================
    nearest_cell_radius = np.min(radius_xy)

    zoom_radius = np.clip(
        ZOOM_RADIUS_FACTOR
        * nearest_cell_radius,
        ZOOM_RADIUS_MIN_AU,
        ZOOM_RADIUS_MAX_AU,
    )

    print(
        f"[INFO] Zoom radius = "
        f"{zoom_radius:.3e} AU"
    )

    # ========================================================
    # 全体図
    # ========================================================
    axis_full = np.linspace(
        DOMAIN_MIN,
        DOMAIN_MAX,
        FULL_RESOLUTION,
    )

    X_full, Y_full = np.meshgrid(
        axis_full,
        axis_full,
    )

    points_xy = points[:, [0, 1]]

    density_full = smooth_interpolation(
        points_xy,
        density,
        X_full,
        Y_full,
    )

    # ========================================================
    # ズーム領域
    # ========================================================
    zoom_mask = (
        radius_xy <= zoom_radius
    )

    points_zoom = points[zoom_mask]
    density_zoom_raw = density[zoom_mask]

    radius_spherical_zoom = (
        radius_spherical[zoom_mask]
    )

    if len(points_zoom) <= 10:
        print(
            f"[WARNING] Insufficient zoom data "
            f"at step {step:05d}"
        )
        continue

    # ========================================================
    # シンク内部を除外する診断用マスク
    # ========================================================
    # 密度マップの補間には全セルを使用するが、
    # extremaとカラースケール計算からはシンク内部を除外する
    outside_sink_mask = (
        radius_spherical_zoom
        >= SINK_MASK_RADIUS_AU
    )

    if not np.any(outside_sink_mask):
        print(
            f"[WARNING] No cells outside sink "
            f"at step {step:05d}"
        )
        continue

    points_diagnostic = points_zoom[
        outside_sink_mask
    ]

    density_diagnostic = density_zoom_raw[
        outside_sink_mask
    ]

    radius_diagnostic = radius_spherical_zoom[
        outside_sink_mask
    ]

    # ========================================================
    # ズーム補間
    # ========================================================
    zoom_axis = np.linspace(
        -zoom_radius,
        zoom_radius,
        ZOOM_RESOLUTION,
    )

    X_zoom, Y_zoom = np.meshgrid(
        zoom_axis,
        zoom_axis,
    )

    # 描画にはシンク内部も含める
    density_zoom = smooth_interpolation(
        points_zoom[:, [0, 1]],
        density_zoom_raw,
        X_zoom,
        Y_zoom,
    )

    # ========================================================
    # ズーム専用カラースケール
    # シンク内部を除外した密度から計算
    # ========================================================
    zoom_vmin, zoom_vmax = log_limits(
        density_diagnostic,
        ZOOM_PERCENTILES,
        fallback=(full_vmin, full_vmax),
    )

    zoom_norm = LogNorm(
        vmin=zoom_vmin,
        vmax=zoom_vmax,
    )

    # ========================================================
    # 最大・最小密度
    # シンク外かつズーム領域内だけを対象にする
    # ========================================================
    maximum_index = np.argmax(
        density_diagnostic
    )

    minimum_index = np.argmin(
        density_diagnostic
    )

    rho_max = density_diagnostic[
        maximum_index
    ]

    rho_min = density_diagnostic[
        minimum_index
    ]

    r_max = radius_diagnostic[
        maximum_index
    ]

    r_min = radius_diagnostic[
        minimum_index
    ]

    position_max = points_diagnostic[
        maximum_index
    ]

    position_min = points_diagnostic[
        minimum_index
    ]

    print(
        f"[INFO] Density extrema outside sink "
        f"(r >= {SINK_MASK_RADIUS_AU:.3e} AU):"
    )

    print(
        f"  rho_max = {rho_max:.3e} "
        f"M_sun AU^-3 at "
        f"({position_max[0]:.3e}, "
        f"{position_max[1]:.3e}, "
        f"{position_max[2]:.3e}) AU, "
        f"r = {r_max:.3e} AU"
    )

    print(
        f"  rho_min = {rho_min:.3e} "
        f"M_sun AU^-3 at "
        f"({position_min[0]:.3e}, "
        f"{position_min[1]:.3e}, "
        f"{position_min[2]:.3e}) AU, "
        f"r = {r_min:.3e} AU"
    )

    

    # Figure
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(
        1,
        5,
        width_ratios=[4, 0.25, 4, 0.25, 1.5],
        wspace=0.35,
    )

    ax_full = fig.add_subplot(gs[0, 0])
    image_full = ax_full.pcolormesh(
        X_full,
        Y_full,
        density_full,
        cmap=FULL_CMAP,
        norm=full_norm,
        shading="auto",
        rasterized=True,
    )

    ax_full.add_patch(
        patches.Rectangle(
            (-zoom_radius, -zoom_radius),
            2.0 * zoom_radius,
            2.0 * zoom_radius,
            edgecolor="cyan",
            facecolor="none",
            linestyle="--",
            linewidth=2,
            label="Zoom region",
        )
    )
    ax_full.plot(
        0.0,
        0.0,
        "r+",
        markersize=12,
        markeredgewidth=2,
        label="Center",
    )
    ax_full.set(
        xlim=(DOMAIN_MIN, DOMAIN_MAX),
        ylim=(DOMAIN_MIN, DOMAIN_MAX),
        xlabel=r"$x\ [{\rm AU}]$",
        ylabel=r"$y\ [{\rm AU}]$",
        title="Full Domain",
        aspect="equal",
    )
    ax_full.ticklabel_format(
        axis="both",
        style="scientific",
        scilimits=(0, 0),
    )
    ax_full.grid(alpha=0.2, linestyle="--")
    ax_full.legend(fontsize=8)

    cax_full = fig.add_subplot(gs[0, 1])
    cbar_full = fig.colorbar(
        image_full,
        cax=cax_full,
        extend="both",
    )
    cbar_full.ax.yaxis.set_major_formatter(
        LogFormatterSciNotation()
    )
    cbar_full.set_label(
        r"$\rho\ [M_\odot\,{\rm AU}^{-3}]$"
        "\nGlobal scale"
    )

    ax_zoom = fig.add_subplot(gs[0, 2])
    ax_zoom.set_aspect("equal")
    
    image_zoom = ax_zoom.imshow(
    density_zoom,
    origin="lower",
    extent=[
        -zoom_radius,
        zoom_radius,
        -zoom_radius,
        zoom_radius,
    ],
    cmap=ZOOM_CMAP,
    norm=zoom_norm,
    interpolation=ZOOM_INTERPOLATION,
    aspect="equal",
    rasterized=True,
    )

    ax_zoom.plot(
        0.0,
        0.0,
        "r+",
        markersize=12,
        markeredgewidth=2,
        label="Center",
    )

    ax_zoom.set_xlim(
        -zoom_radius,
        zoom_radius,
    )

    ax_zoom.set_ylim(
        -zoom_radius,
        zoom_radius,
    )

    ax_zoom.set_xlabel(
        r"$x\ [{\rm AU}]$",
        fontsize=12,
    )

    ax_zoom.set_ylabel(
        r"$y\ [{\rm AU}]$",
        fontsize=12,
    )

    ax_zoom.set_title(
        "Zoomed Region\n"
        f"R = {zoom_radius:.3e} AU",
        fontsize=12,
        fontweight="bold",
    )

    ax_zoom.ticklabel_format(
        axis="both",
        style="scientific",
        scilimits=(0, 0),
    )

    # 別添ズーム版と同様、通常のグリッド線も非表示
    ax_zoom.grid(False)

    ax_zoom.legend(
        loc="upper right",
        fontsize=8,
        framealpha=0.7,
    )

    if density_zoom is not None:
        image_zoom = ax_zoom.imshow(
            density_zoom,
            origin="lower",
            extent=[
                -zoom_radius,
                zoom_radius,
                -zoom_radius,
                zoom_radius,
            ],
            cmap=ZOOM_CMAP,
            norm=zoom_norm,
            interpolation=ZOOM_INTERPOLATION,
            aspect="equal",
            rasterized=True,
        )

    ax_zoom.plot(
        0.0,
        0.0,
        "r+",
        markersize=12,
        markeredgewidth=2,
        label="Center",
    )
    ax_zoom.set(
        xlim=(-zoom_radius, zoom_radius),
        ylim=(-zoom_radius, zoom_radius),
        xlabel=r"$x\ [{\rm AU}]$",
        ylabel=r"$y\ [{\rm AU}]$",
        title=(
            "Zoomed Region\n"
            f"R = {zoom_radius:.3e} AU"
        ),
        aspect="equal",
    )
    ax_zoom.ticklabel_format(
        axis="both",
        style="scientific",
        scilimits=(0, 0),
    )
    ax_zoom.grid(alpha=0.15, linestyle="--")
    ax_zoom.legend(fontsize=8)

    cax_zoom = fig.add_subplot(gs[0, 3])

    if image_zoom is not None:
        cbar_zoom = fig.colorbar(
            image_zoom,
            cax=cax_zoom,
            extend="both",
        )
        cbar_zoom.ax.yaxis.set_major_formatter(
            LogFormatterSciNotation()
        )
        cbar_zoom.set_label(
        r"$\rho\ [M_\odot\,{\rm AU}^{-3}]$"
        "\n"
        f"Zoom {ZOOM_PERCENTILES[0]:.0f}–"
        f"{ZOOM_PERCENTILES[1]:.0f} percentile"
    )
    else:
        cax_zoom.axis("off")

    ax_info = fig.add_subplot(gs[0, 4])
    ax_info.axis("off")
    ax_info.text(
        0.04,
        0.95,
        (
            f"Time\n{time_yr:.6e} yr\n\n"
            f"Code time\n{time_code:.6e}\n\n"
            f"Density extrema\n"
            f"(outside sink)\n"
            f"r >= {SINK_MASK_RADIUS_AU:.3e} AU\n\n"
            f"Maximum density\n"
            f"{rho_max:.3e}\n"
            r"$M_\odot\,{\rm AU}^{-3}$"
            f"\nr = {r_max:.3e} AU\n\n"
            f"Minimum density\n"
            f"{rho_min:.3e}\n"
            r"$M_\odot\,{\rm AU}^{-3}$"
            f"\nr = {r_min:.3e} AU\n\n"
            f"Zoom color range\n"
            f"{zoom_vmin:.2e} to {zoom_vmax:.2e}"
        ),
        transform=ax_info.transAxes,
        va="top",
        fontsize=10,
        bbox=dict(
            boxstyle="round",
            facecolor="whitesmoke",
            edgecolor="black",
            alpha=0.9,
        ),
    )

    fig.suptitle(
        "Density Map: x-y Midplane\n"
        f"t = {time_yr:.6e} yr | "
        f"Variable: {density_name}",
        fontsize=14,
        fontweight="bold",
    )

    output_file = os.path.join(
        output_dir,
        f"density_xy_timestep_{step:05d}_no_grid.png",
    )

    fig.savefig(
        output_file,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)

    print(
        f"[INFO] {index + 1}/{len(steps)}: "
        f"step={step:05d}, "
        f"time={time_yr:.6e} yr"
    )

print(f"[INFO] Saved to: {output_dir}")

# ============================================================
# 密度マップ（x-z平面・AMRグリッドなし）
#
# 表示単位
#   距離：AU
#   質量：M_sun
#   密度：M_sun AU^-3
#
# 仕様
#   1. y=0と交差するAMRブロックのみ使用
#   2. 各ブロックからy=0に最も近い1層だけ抽出
#   3. AMR境界線は表示しない
#   4. 線形補間＋最近傍補間で穴埋め
#   5. ズーム図はbilinear表示
#   6. ズーム半径：2×10^4～5×10^4 AU
#   7. ズーム色範囲：シンク外密度の5～98パーセンタイル
#   8. 最大・最小密度：シンク外かつズーム領域内で検索
#   9. 密度マップ自体にはシンク内部も表示
#  10. VTKヘッダーのコード時刻をyrへ変換
#  11. ファイル名はタイムステップ順
# ============================================================

import os
import re
from collections import defaultdict

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import LogFormatterSciNotation
from scipy.interpolate import griddata


# ============================================================
# 入出力設定
# ============================================================
vtk_dir = os.path.expanduser(
    "~/athena-project/results/〇〇"
)

output_dir = "./xz_density_maps_no_grid"
os.makedirs(output_dir, exist_ok=True)


# ============================================================
# 単位定義
# ============================================================
# Toyouchi.cppのコード単位
M_UNIT_CGS = 4.0e33   # g
L_UNIT_CGS = 6.7e15   # cm
T_UNIT_CGS = 3.34e10  # s

# 物理定数
AU_CGS = 1.495978707e13
MSUN_CGS = 1.98847e33
YEAR_CGS = 365.25 * 24.0 * 3600.0

# コード単位 → 表示単位
LENGTH_UNIT_AU = L_UNIT_CGS / AU_CGS
MASS_UNIT_MSUN = M_UNIT_CGS / MSUN_CGS
TIME_UNIT_YR = T_UNIT_CGS / YEAR_CGS

# code density → M_sun AU^-3
DENSITY_UNIT = (
    MASS_UNIT_MSUN / LENGTH_UNIT_AU**3
)

print("[INFO] Unit conversion factors:")
print(
    f"  1 code length  = "
    f"{LENGTH_UNIT_AU:.6e} AU"
)
print(
    f"  1 code mass    = "
    f"{MASS_UNIT_MSUN:.6e} M_sun"
)
print(
    f"  1 code time    = "
    f"{TIME_UNIT_YR:.6e} yr"
)
print(
    f"  1 code density = "
    f"{DENSITY_UNIT:.6e} M_sun AU^-3"
)


# ============================================================
# 計算領域
# ============================================================
# 新しい入力ファイル：
# x1, x2, x3 = -224～+224 code length
LBOX_CODE = 448.0

LBOX_AU = (
    LBOX_CODE * LENGTH_UNIT_AU
)

DOMAIN_MIN = -LBOX_AU / 2.0
DOMAIN_MAX = LBOX_AU / 2.0


# ============================================================
# 描画・ズーム設定
# ============================================================
FULL_RESOLUTION = 800
ZOOM_RESOLUTION = 800

# 自動ズーム倍率
ZOOM_RADIUS_FACTOR = 20.0

# 新しい計算領域用
# AU単位で直接指定
ZOOM_RADIUS_MIN_AU = 2.0e4
ZOOM_RADIUS_MAX_AU = 5.0e4

# 全体図の色範囲
FULL_PERCENTILES = (1.0, 99.0)

# ズーム図の色範囲
ZOOM_PERCENTILES = (5.0, 98.0)

FULL_CMAP = "inferno"
ZOOM_CMAP = "turbo"
ZOOM_INTERPOLATION = "bilinear"

dpi = 200
figsize = (19, 8)


# ============================================================
# シンクマスク設定
# ============================================================
# inputファイルのr_sink_auと一致させる
SINK_RADIUS_AU = 1000.0

# シンク境界付近も除外する場合は
# 1.1～1.2程度に変更
SINK_MASK_FACTOR = 1.0

SINK_MASK_RADIUS_AU = (
    SINK_MASK_FACTOR * SINK_RADIUS_AU
)

print("[INFO] Domain and mask settings:")
print(
    f"  Domain width = {LBOX_AU:.6e} AU"
)
print(
    f"  Sink mask    = "
    f"r < {SINK_MASK_RADIUS_AU:.6e} AU"
)
print(
    f"  Zoom range   = "
    f"{ZOOM_RADIUS_MIN_AU:.6e}–"
    f"{ZOOM_RADIUS_MAX_AU:.6e} AU"
)


# ============================================================
# VTKヘッダーからコード時刻を取得
# ============================================================
def read_vtk_time_code(filename):
    with open(filename, "rb") as vtk_file:
        header = vtk_file.read(512).decode(
            "ascii",
            errors="ignore",
        )

    match = re.search(
        r"time\s*=\s*"
        r"([+-]?(?:\d+\.?\d*|\.\d+)"
        r"(?:[eE][+-]?\d+)?)",
        header,
    )

    if match is None:
        raise ValueError(
            f"Could not find time in VTK header: "
            f"{filename}"
        )

    return float(match.group(1))


# ============================================================
# 対数カラースケール
# ============================================================
def calculate_log_limits(
    values,
    percentiles,
    fallback=None,
):
    values = np.asarray(values).ravel()

    positive_values = values[
        np.isfinite(values)
        & (values > 0.0)
    ]

    if len(positive_values) == 0:
        if fallback is not None:
            return fallback

        return 1.0e-17, 1.0e-12

    vmin, vmax = np.percentile(
        positive_values,
        percentiles,
    )

    if not np.isfinite(vmin) or vmin <= 0.0:
        vmin = np.min(positive_values)

    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = np.max(positive_values)

    if vmax <= vmin:
        vmin *= 0.5
        vmax *= 2.0

    # 色範囲が狭すぎる場合は最低1桁確保
    if vmax / vmin < 10.0:
        center = np.sqrt(vmin * vmax)

        vmin = center / np.sqrt(10.0)
        vmax = center * np.sqrt(10.0)

    return vmin, vmax


# ============================================================
# y=0に最も近いセル層を抽出
# ============================================================
def extract_xz_midplane(
    grid,
    density_name,
):
    """
    y=0と交差するAMRブロックから、
    y=0に最も近いセル中心面を1層だけ抽出する。

    Returns
    -------
    points_au
        (x, y, z)座標。単位はAU。
    density
        密度。単位はM_sun AU^-3。
    """
    bounds = grid.bounds

    # bounds = (xmin, xmax, ymin, ymax, zmin, zmax)
    if not (
        bounds[2] <= 0.0 <= bounds[3]
    ):
        return None, None

    points_code = (
        grid.cell_centers().points
    )

    if len(points_code) == 0:
        return None, None

    density_code = np.asarray(
        grid[density_name]
    ).reshape(-1)

    if len(density_code) != len(points_code):
        raise ValueError(
            "Density size does not match "
            "cell-center size."
        )

    y_values = points_code[:, 1]

    y_nearest = y_values[
        np.argmin(np.abs(y_values))
    ]

    tolerance = (
        1.0e-10
        * max(np.max(np.abs(y_values)), 1.0)
    )

    midplane_mask = np.isclose(
        y_values,
        y_nearest,
        rtol=1.0e-10,
        atol=tolerance,
    )

    if not np.any(midplane_mask):
        return None, None

    points_au = (
        points_code[midplane_mask]
        * LENGTH_UNIT_AU
    )

    density = (
        density_code[midplane_mask]
        * DENSITY_UNIT
    )

    return points_au, density


# ============================================================
# 線形補間＋最近傍補間
# ============================================================
def smooth_interpolation(
    points_2d,
    values,
    X,
    Z,
):
    """
    線形補間を基本とし、
    線形補間できない外縁だけを最近傍補間で埋める。
    """
    linear = griddata(
        points_2d,
        values,
        (X, Z),
        method="linear",
    )

    nearest = griddata(
        points_2d,
        values,
        (X, Z),
        method="nearest",
    )

    result = np.where(
        np.isfinite(linear),
        linear,
        nearest,
    )

    invalid = (
        ~np.isfinite(result)
        | (result <= 0.0)
    )

    result[invalid] = np.nan

    return result


# ============================================================
# VTKファイル整理
# ============================================================
if not os.path.isdir(vtk_dir):
    raise FileNotFoundError(
        f"VTK directory does not exist: {vtk_dir}"
    )

files_by_step = defaultdict(list)

for filename in os.listdir(vtk_dir):
    if not (
        filename.startswith("Toyouchi.block")
        and filename.endswith(".vtk")
    ):
        continue

    match = re.search(
        r"(?:prim\.)?out2\.(\d+)",
        filename,
    )

    if match is not None:
        timestep = int(match.group(1))

        files_by_step[timestep].append(
            os.path.join(
                vtk_dir,
                filename,
            )
        )

steps = sorted(files_by_step)

if not steps:
    raise RuntimeError(
        f"No VTK files found in {vtk_dir}"
    )

print(
    f"[INFO] Found {len(steps)} timesteps"
)
print(
    f"[INFO] Timestep range: "
    f"{steps[0]}–{steps[-1]}"
)


# ============================================================
# 密度変数名検出
# ============================================================
test_file = files_by_step[steps[0]][0]
test_grid = pv.read(test_file)

print(
    f"[INFO] Available arrays: "
    f"{test_grid.array_names}"
)

density_name = next(
    (
        name
        for name in [
            "dens",
            "density",
            "rho",
            "prim_dens",
            "prim_density",
        ]
        if name in test_grid.array_names
    ),
    None,
)

if density_name is None:
    raise RuntimeError(
        "No density array was found."
    )

print(
    f"[INFO] Density variable: "
    f"{density_name}"
)


# ============================================================
# 全体図共通カラースケール
# シンク内部を除外した密度から計算
# ============================================================
sample_indices = np.linspace(
    0,
    len(steps) - 1,
    min(10, len(steps)),
).astype(int)

sample_steps = [
    steps[index]
    for index in sample_indices
]

sample_density = []

for step in sample_steps:
    for filename in files_by_step[step]:
        try:
            grid = pv.read(filename)

            sample_points, density_sample = (
                extract_xz_midplane(
                    grid,
                    density_name,
                )
            )

            if sample_points is None:
                continue

            sample_radius_spherical = (
                np.linalg.norm(
                    sample_points,
                    axis=1,
                )
            )

            sample_outside_sink = (
                sample_radius_spherical
                >= SINK_MASK_RADIUS_AU
            )

            if np.any(sample_outside_sink):
                sample_density.append(
                    density_sample[
                        sample_outside_sink
                    ]
                )

        except Exception as error:
            print(
                f"[WARNING] Could not sample "
                f"{filename}: {error}"
            )

if not sample_density:
    raise RuntimeError(
        "No valid density data outside "
        "the sink were found."
    )

sample_density = np.concatenate(
    sample_density
)

full_vmin, full_vmax = (
    calculate_log_limits(
        sample_density,
        FULL_PERCENTILES,
    )
)

full_norm = LogNorm(
    vmin=full_vmin,
    vmax=full_vmax,
)

print(
    f"[INFO] Global density range "
    f"(outside sink): "
    f"{full_vmin:.3e}–"
    f"{full_vmax:.3e} M_sun AU^-3"
)


# ============================================================
# メイン処理
# ============================================================
for timestep_index, step in enumerate(steps):
    print(
        f"[INFO] Processing timestep "
        f"{step:05d} "
        f"({timestep_index + 1}/"
        f"{len(steps)})"
    )

    point_arrays = []
    density_arrays = []

    for filename in files_by_step[step]:
        try:
            grid = pv.read(filename)

            points, density = (
                extract_xz_midplane(
                    grid,
                    density_name,
                )
            )

            if points is not None:
                point_arrays.append(points)
                density_arrays.append(density)

        except Exception as error:
            print(
                f"[WARNING] Failed to process "
                f"{filename}: {error}"
            )

    if not point_arrays:
        print(
            f"[WARNING] No x-z midplane data "
            f"at timestep {step:05d}"
        )
        continue

    points = np.vstack(point_arrays)
    density = np.hstack(density_arrays)

    valid = (
        np.all(
            np.isfinite(points),
            axis=1,
        )
        & np.isfinite(density)
        & (density > 0.0)
    )

    points = points[valid]
    density = density[valid]

    if len(density) == 0:
        print(
            f"[WARNING] No valid density data "
            f"at timestep {step:05d}"
        )
        continue

    # ========================================================
    # 時刻
    # ========================================================
    representative_file = (
        files_by_step[step][0]
    )

    time_code = read_vtk_time_code(
        representative_file
    )

    time_yr = (
        time_code * TIME_UNIT_YR
    )

    # ========================================================
    # 平面内半径と球半径
    # ========================================================
    radius_xz = np.hypot(
        points[:, 0],
        points[:, 2],
    )

    # シンク判定には球半径を使用
    radius_spherical = np.linalg.norm(
        points,
        axis=1,
    )

    # ========================================================
    # ズーム半径
    # ========================================================
    nearest_cell_radius = np.min(
        radius_xz
    )

    zoom_radius = np.clip(
        ZOOM_RADIUS_FACTOR
        * nearest_cell_radius,
        ZOOM_RADIUS_MIN_AU,
        ZOOM_RADIUS_MAX_AU,
    )

    # 実データ領域を超えないように制限
    data_half_width = min(
        np.max(np.abs(points[:, 0])),
        np.max(np.abs(points[:, 2])),
    )

    zoom_radius = min(
        zoom_radius,
        data_half_width,
    )

    print(
        f"[INFO] Zoom radius = "
        f"{zoom_radius:.3e} AU"
    )

    # ========================================================
    # 全体図補間
    # ========================================================
    axis_full = np.linspace(
        DOMAIN_MIN,
        DOMAIN_MAX,
        FULL_RESOLUTION,
    )

    X_full, Z_full = np.meshgrid(
        axis_full,
        axis_full,
    )

    density_full = smooth_interpolation(
        points[:, [0, 2]],
        density,
        X_full,
        Z_full,
    )

    # ========================================================
    # ズーム領域
    # ========================================================
    zoom_mask = (
        radius_xz <= zoom_radius
    )

    points_zoom = points[zoom_mask]
    density_zoom_raw = density[zoom_mask]

    radius_spherical_zoom = (
        radius_spherical[zoom_mask]
    )

    if len(points_zoom) <= 10:
        print(
            f"[WARNING] Insufficient zoom data "
            f"at timestep {step:05d}"
        )
        continue

    # ========================================================
    # シンク外部を選ぶ診断用マスク
    # ========================================================
    outside_sink_mask = (
        radius_spherical_zoom
        >= SINK_MASK_RADIUS_AU
    )

    if not np.any(outside_sink_mask):
        print(
            f"[WARNING] No cells outside sink "
            f"at timestep {step:05d}"
        )
        continue

    points_diagnostic = points_zoom[
        outside_sink_mask
    ]

    density_diagnostic = density_zoom_raw[
        outside_sink_mask
    ]

    radius_diagnostic = (
        radius_spherical_zoom[
            outside_sink_mask
        ]
    )

    # ========================================================
    # ズーム補間
    # 描画にはシンク内部も含める
    # ========================================================
    axis_zoom = np.linspace(
        -zoom_radius,
        zoom_radius,
        ZOOM_RESOLUTION,
    )

    X_zoom, Z_zoom = np.meshgrid(
        axis_zoom,
        axis_zoom,
    )

    density_zoom = smooth_interpolation(
        points_zoom[:, [0, 2]],
        density_zoom_raw,
        X_zoom,
        Z_zoom,
    )

    # ========================================================
    # ズーム専用カラースケール
    # シンク外部の密度のみから計算
    # ========================================================
    zoom_vmin, zoom_vmax = (
        calculate_log_limits(
            density_diagnostic,
            ZOOM_PERCENTILES,
            fallback=(
                full_vmin,
                full_vmax,
            ),
        )
    )

    zoom_norm = LogNorm(
        vmin=zoom_vmin,
        vmax=zoom_vmax,
    )

    # ========================================================
    # 最大・最小密度
    # シンク外かつズーム領域内だけを検索
    # ========================================================
    maximum_index = np.argmax(
        density_diagnostic
    )

    minimum_index = np.argmin(
        density_diagnostic
    )

    rho_max = density_diagnostic[
        maximum_index
    ]

    rho_min = density_diagnostic[
        minimum_index
    ]

    r_max = radius_diagnostic[
        maximum_index
    ]

    r_min = radius_diagnostic[
        minimum_index
    ]

    position_max = points_diagnostic[
        maximum_index
    ]

    position_min = points_diagnostic[
        minimum_index
    ]

    print(
        f"[INFO] Density extrema outside sink "
        f"(r >= {SINK_MASK_RADIUS_AU:.3e} AU):"
    )

    print(
        f"  rho_max = {rho_max:.3e} "
        f"M_sun AU^-3 at "
        f"({position_max[0]:.3e}, "
        f"{position_max[1]:.3e}, "
        f"{position_max[2]:.3e}) AU, "
        f"r = {r_max:.3e} AU"
    )

    print(
        f"  rho_min = {rho_min:.3e} "
        f"M_sun AU^-3 at "
        f"({position_min[0]:.3e}, "
        f"{position_min[1]:.3e}, "
        f"{position_min[2]:.3e}) AU, "
        f"r = {r_min:.3e} AU"
    )

    # ========================================================
    # Figure
    # ========================================================
    fig = plt.figure(figsize=figsize)

    gs = GridSpec(
        1,
        5,
        width_ratios=[
            4.0,
            0.25,
            4.0,
            0.25,
            1.6,
        ],
        wspace=0.35,
    )

    # ========================================================
    # 全体図
    # ========================================================
    ax_full = fig.add_subplot(gs[0, 0])
    ax_full.set_aspect("equal")

    image_full = ax_full.pcolormesh(
        X_full,
        Z_full,
        density_full,
        cmap=FULL_CMAP,
        norm=full_norm,
        shading="auto",
        rasterized=True,
    )

    # ズーム領域
    zoom_rectangle = patches.Rectangle(
        (
            -zoom_radius,
            -zoom_radius,
        ),
        2.0 * zoom_radius,
        2.0 * zoom_radius,
        edgecolor="cyan",
        facecolor="none",
        linestyle="--",
        linewidth=2.0,
        label="Zoom region",
    )

    ax_full.add_patch(
        zoom_rectangle
    )

    ax_full.plot(
        0.0,
        0.0,
        "r+",
        markersize=12,
        markeredgewidth=2,
        label="Center",
    )

    ax_full.set_xlim(
        DOMAIN_MIN,
        DOMAIN_MAX,
    )

    ax_full.set_ylim(
        DOMAIN_MIN,
        DOMAIN_MAX,
    )

    ax_full.set_xlabel(
        r"$x\ [{\rm AU}]$",
        fontsize=12,
    )

    ax_full.set_ylabel(
        r"$z\ [{\rm AU}]$",
        fontsize=12,
    )

    ax_full.set_title(
        "Full Domain",
        fontsize=12,
        fontweight="bold",
    )

    ax_full.ticklabel_format(
        axis="both",
        style="scientific",
        scilimits=(0, 0),
    )

    ax_full.grid(False)

    ax_full.legend(
        loc="upper right",
        fontsize=8,
        framealpha=0.75,
    )

    # ========================================================
    # 全体図カラーバー
    # ========================================================
    cax_full = fig.add_subplot(gs[0, 1])

    cbar_full = fig.colorbar(
        image_full,
        cax=cax_full,
        extend="both",
    )

    cbar_full.ax.yaxis.set_major_formatter(
        LogFormatterSciNotation()
    )

    cbar_full.set_label(
        r"$\rho\ "
        r"[M_\odot\,{\rm AU}^{-3}]$"
        "\nGlobal scale"
        "\n(outside sink)",
        fontsize=10,
    )

    # ========================================================
    # ズーム図
    # ========================================================
    ax_zoom = fig.add_subplot(gs[0, 2])
    ax_zoom.set_aspect("equal")

    image_zoom = ax_zoom.imshow(
        density_zoom,
        origin="lower",
        extent=[
            -zoom_radius,
            zoom_radius,
            -zoom_radius,
            zoom_radius,
        ],
        cmap=ZOOM_CMAP,
        norm=zoom_norm,
        interpolation=ZOOM_INTERPOLATION,
        aspect="equal",
        rasterized=True,
    )

    ax_zoom.plot(
        0.0,
        0.0,
        "r+",
        markersize=12,
        markeredgewidth=2,
        label="Center",
    )

    ax_zoom.set_xlim(
        -zoom_radius,
        zoom_radius,
    )

    ax_zoom.set_ylim(
        -zoom_radius,
        zoom_radius,
    )

    ax_zoom.set_xlabel(
        r"$x\ [{\rm AU}]$",
        fontsize=12,
    )

    ax_zoom.set_ylabel(
        r"$z\ [{\rm AU}]$",
        fontsize=12,
    )

    ax_zoom.set_title(
        "Zoomed Region\n"
        f"R = {zoom_radius:.3e} AU",
        fontsize=12,
        fontweight="bold",
    )

    ax_zoom.ticklabel_format(
        axis="both",
        style="scientific",
        scilimits=(0, 0),
    )

    ax_zoom.grid(False)

    ax_zoom.legend(
        loc="upper right",
        fontsize=8,
        framealpha=0.7,
    )

    # ========================================================
    # ズーム図カラーバー
    # ========================================================
    cax_zoom = fig.add_subplot(gs[0, 3])

    cbar_zoom = fig.colorbar(
        image_zoom,
        cax=cax_zoom,
        extend="both",
    )

    cbar_zoom.ax.yaxis.set_major_formatter(
        LogFormatterSciNotation()
    )

    cbar_zoom.set_label(
        r"$\rho\ "
        r"[M_\odot\,{\rm AU}^{-3}]$"
        "\n"
        f"Zoom {ZOOM_PERCENTILES[0]:.0f}–"
        f"{ZOOM_PERCENTILES[1]:.0f} percentile"
        "\n(outside sink)",
        fontsize=10,
    )

    # ========================================================
    # 情報欄
    # ========================================================
    ax_info = fig.add_subplot(gs[0, 4])
    ax_info.axis("off")

    information_text = (
        "Time\n"
        f"{time_yr:.6e} yr\n\n"

        "Code time\n"
        f"{time_code:.6e}\n\n"

        "Zoom radius\n"
        f"{zoom_radius:.3e} AU\n\n"

        "Density search region\n"
        f"r >= {SINK_MASK_RADIUS_AU:.3e} AU\n"
        "(outside sink)\n\n"

        "Maximum density\n"
        f"{rho_max:.3e}\n"
        r"$M_\odot\,{\rm AU}^{-3}$"
        "\n"
        f"r = {r_max:.3e} AU\n\n"

        "Minimum density\n"
        f"{rho_min:.3e}\n"
        r"$M_\odot\,{\rm AU}^{-3}$"
        "\n"
        f"r = {r_min:.3e} AU\n\n"

        "Zoom color range\n"
        f"{zoom_vmin:.2e}\n"
        "to\n"
        f"{zoom_vmax:.2e}"
    )

    ax_info.text(
        0.04,
        0.95,
        information_text,
        transform=ax_info.transAxes,
        fontsize=9,
        verticalalignment="top",
        bbox=dict(
            boxstyle="round",
            facecolor="whitesmoke",
            edgecolor="black",
            alpha=0.9,
        ),
    )

    # ========================================================
    # タイトル
    # ========================================================
    fig.suptitle(
        "Density Map: x-z Midplane\n"
        f"t = {time_yr:.6e} yr | "
        f"Variable: {density_name}",
        fontsize=14,
        fontweight="bold",
        y=0.98,
    )

    plt.subplots_adjust(
        top=0.88,
        bottom=0.10,
        left=0.05,
        right=0.97,
    )

    # ========================================================
    # タイムステップ順のファイル名で保存
    # ========================================================
    png = os.path.join(
        output_dir,
        (
            f"density_xz_timestep_"
            f"{step:05d}_"
            f"time_{time_yr:.6e}yr_"
            "no_grid.png"
        ),
    )

    fig.savefig(
        png,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)

    print(
        f"[INFO] Saved: "
        f"step={step:05d}, "
        f"time={time_yr:.6e} yr"
    )


print(
    f"[INFO] All x-z density maps "
    f"saved to: {output_dir}"
)

# ============================================================
# x-y平面・ズーム密度マップのみ
#
# ・AMR境界線なし
# ・通常のグリッド線なし
# ・z=0に最も近いセル層を1層だけ使用
# ・距離：AU
# ・密度：M_sun AU^-3
# ・VTKヘッダーのコード時刻をyrへ変換
# ・線形補間＋最近傍補間による穴埋め
# ・bilinear表示
# ・ズーム領域内の1–99パーセンタイルで色範囲を調整
# ・ファイル名はタイムステップ順
# ============================================================

import os
import re
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import LogFormatterSciNotation
from scipy.interpolate import griddata


# ============================================================
# 入出力設定
# ============================================================
vtk_dir = os.path.expanduser(
    "~/athena-project/results/〇〇"
)

output_dir = "./xy_density_zoom_no_grid"
os.makedirs(output_dir, exist_ok=True)


# ============================================================
# 単位定義
# ============================================================
# Toyouchi.cppのコード単位
M_UNIT_CGS = 4.0e33   # g
L_UNIT_CGS = 6.7e15   # cm
T_UNIT_CGS = 3.34e10  # s

# 物理定数
AU_CGS = 1.495978707e13
MSUN_CGS = 1.98847e33
YEAR_CGS = 365.25 * 24.0 * 3600.0

# コード単位 → 表示単位
LENGTH_UNIT_AU = L_UNIT_CGS / AU_CGS
MASS_UNIT_MSUN = M_UNIT_CGS / MSUN_CGS
TIME_UNIT_YR = T_UNIT_CGS / YEAR_CGS

# code density → M_sun AU^-3
DENSITY_UNIT_MSUN_AU3 = (
    MASS_UNIT_MSUN / LENGTH_UNIT_AU**3
)

print("[INFO] Unit conversion factors:")
print(
    f"  1 code length  = "
    f"{LENGTH_UNIT_AU:.6e} AU"
)
print(
    f"  1 code mass    = "
    f"{MASS_UNIT_MSUN:.6e} M_sun"
)
print(
    f"  1 code time    = "
    f"{TIME_UNIT_YR:.6e} yr"
)
print(
    f"  1 code density = "
    f"{DENSITY_UNIT_MSUN_AU3:.6e} "
    "M_sun AU^-3"
)


# ============================================================
# ズーム設定
# ============================================================
ZOOM_RESOLUTION = 800
ZOOM_RADIUS_FACTOR = 20.0


# 計算領域用のズーム半径
# AU単位で直接指定する
ZOOM_RADIUS_MIN_AU = 2.0e4   # 最小ズーム半径：2,0000 AU
ZOOM_RADIUS_MAX_AU = 5.0e4   # 最大ズーム半径：50,000 AU



# ズーム領域内の色範囲
ZOOM_COLOR_PERCENTILES = (5.0, 98.0)

ZOOM_CMAP = "turbo"
ZOOM_IMAGE_INTERPOLATION = "bilinear"

dpi = 200
figsize = (10, 8)

# ============================================================
# シンク領域のマスク設定
# inputファイルのr_sink_auと一致させる
# ============================================================
SINK_RADIUS_AU = 1000.0

# シンク境界直近まで除外する場合は1.1～1.2に変更
SINK_MASK_FACTOR = 1.0

SINK_MASK_RADIUS_AU = (
    SINK_MASK_FACTOR * SINK_RADIUS_AU
)


# ============================================================
# VTKヘッダーから時刻を取得
# ============================================================
def read_vtk_time_code(filename):
    """
    VTKヘッダーの
    '# Athena++ data at time=...'
    からコード時刻を読み取る。
    """
    with open(filename, "rb") as vtk_file:
        header = vtk_file.read(512).decode(
            "ascii",
            errors="ignore",
        )

    match = re.search(
        r"time\s*=\s*"
        r"([+-]?(?:\d+\.?\d*|\.\d+)"
        r"(?:[eE][+-]?\d+)?)",
        header,
    )

    if match is None:
        raise ValueError(
            f"Could not find time in VTK header: "
            f"{filename}"
        )

    return float(match.group(1))


def read_vtk_time_yr(filename):
    return (
        read_vtk_time_code(filename)
        * TIME_UNIT_YR
    )


# ============================================================
# 対数カラースケール
# ============================================================
def calculate_log_limits(
    values,
    lower_percentile=1.0,
    upper_percentile=99.0,
):
    values = np.asarray(values).ravel()

    positive_values = values[
        np.isfinite(values)
        & (values > 0.0)
    ]

    if len(positive_values) == 0:
        raise ValueError(
            "No positive finite density values."
        )

    vmin = np.percentile(
        positive_values,
        lower_percentile,
    )
    vmax = np.percentile(
        positive_values,
        upper_percentile,
    )

    if not np.isfinite(vmin) or vmin <= 0.0:
        vmin = np.min(positive_values)

    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = np.max(positive_values)

    if vmax <= vmin:
        vmin *= 0.5
        vmax *= 2.0

    # 最低1桁の色範囲を確保
    if vmax / vmin < 10.0:
        center = np.sqrt(vmin * vmax)
        vmin = center / np.sqrt(10.0)
        vmax = center * np.sqrt(10.0)

    return vmin, vmax


# ============================================================
# z=0に最も近いセル層を抽出
# ============================================================
def extract_xy_midplane(
    grid,
    density_name,
):
    """
    z=0と交差するブロックから、z=0に最も近いセル中心面を
    1層だけ抽出する。
    """
    bounds = grid.bounds

    # bounds = (xmin, xmax, ymin, ymax, zmin, zmax)
    if not (
        bounds[4] <= 0.0 <= bounds[5]
    ):
        return None, None

    points_code = grid.cell_centers().points

    if len(points_code) == 0:
        return None, None

    density_code = np.asarray(
        grid[density_name]
    ).reshape(-1)

    if len(density_code) != len(points_code):
        raise ValueError(
            "Density size does not match "
            "cell-center size."
        )

    z_values = points_code[:, 2]

    z_nearest = z_values[
        np.argmin(np.abs(z_values))
    ]

    tolerance = (
        1.0e-10
        * max(np.max(np.abs(z_values)), 1.0)
    )

    midplane_mask = np.isclose(
        z_values,
        z_nearest,
        rtol=1.0e-10,
        atol=tolerance,
    )

    if not np.any(midplane_mask):
        return None, None

    points_au = (
        points_code[midplane_mask]
        * LENGTH_UNIT_AU
    )

    density_msun_au3 = (
        density_code[midplane_mask]
        * DENSITY_UNIT_MSUN_AU3
    )

    return points_au, density_msun_au3


# ============================================================
# 線形補間＋最近傍補間
# ============================================================
def interpolate_density_smooth(
    points_2d,
    density,
    X,
    Y,
):
    # 主補間
    density_linear = griddata(
        points_2d,
        density,
        (X, Y),
        method="linear",
    )

    # 線形補間できない外縁用
    density_nearest = griddata(
        points_2d,
        density,
        (X, Y),
        method="nearest",
    )

    density_combined = np.where(
        np.isfinite(density_linear),
        density_linear,
        density_nearest,
    )

    invalid = (
        ~np.isfinite(density_combined)
        | (density_combined <= 0.0)
    )
    density_combined[invalid] = np.nan

    return density_combined


# ============================================================
# VTKファイル整理
# ============================================================
if not os.path.isdir(vtk_dir):
    raise FileNotFoundError(
        f"VTK directory does not exist: {vtk_dir}"
    )

files_by_step = defaultdict(list)

for filename in os.listdir(vtk_dir):
    if not (
        filename.startswith("Toyouchi.block")
        and filename.endswith(".vtk")
    ):
        continue

    match = re.search(
        r"(?:prim\.)?out2\.(\d+)",
        filename,
    )

    if match is not None:
        timestep = int(match.group(1))

        files_by_step[timestep].append(
            os.path.join(
                vtk_dir,
                filename,
            )
        )

timesteps = sorted(files_by_step)

if not timesteps:
    raise RuntimeError(
        f"No VTK files found in {vtk_dir}"
    )

print(
    f"[INFO] Found {len(timesteps)} timesteps"
)
print(
    f"[INFO] Timestep range: "
    f"{timesteps[0]}–{timesteps[-1]}"
)


# ============================================================
# 密度変数名検出
# ============================================================
test_file = files_by_step[timesteps[0]][0]
test_grid = pv.read(test_file)

density_name = next(
    (
        name
        for name in [
            "dens",
            "density",
            "rho",
            "prim_dens",
            "prim_density",
        ]
        if name in test_grid.array_names
    ),
    None,
)

if density_name is None:
    raise RuntimeError(
        "No density array was found."
    )

print(
    f"[INFO] Density variable: {density_name}"
)


# ============================================================
# メイン処理
# ============================================================
for timestep_index, timestep in enumerate(
    timesteps
):
    print(
        f"[INFO] Processing timestep "
        f"{timestep:05d} "
        f"({timestep_index + 1}/"
        f"{len(timesteps)})"
    )

    points_list = []
    density_list = []

    for filename in files_by_step[timestep]:
        try:
            grid = pv.read(filename)

            points_au, density = (
                extract_xy_midplane(
                    grid,
                    density_name,
                )
            )

            if points_au is not None:
                points_list.append(points_au)
                density_list.append(density)

        except Exception as error:
            print(
                f"[WARNING] {filename}: {error}"
            )

    if not points_list:
        print(
            f"[WARNING] No x-y midplane data "
            f"at timestep {timestep:05d}"
        )
        continue

    points_au = np.vstack(points_list)
    density = np.hstack(density_list)

    valid = (
        np.all(
            np.isfinite(points_au),
            axis=1,
        )
        & np.isfinite(density)
        & (density > 0.0)
    )

    points_au = points_au[valid]
    density = density[valid]

    if len(density) == 0:
        continue

    # 物理時刻
    representative_file = (
        files_by_step[timestep][0]
    )

    time_code = read_vtk_time_code(
        representative_file
    )
    time_yr = time_code * TIME_UNIT_YR

    # x-y平面内半径
    radius_xy = np.hypot(
        points_au[:, 0],
        points_au[:, 1],
    )
    
    # シンク判定用の球半径
    # シミュレーション本体のr_sphと同じ定義
    radius_spherical = np.linalg.norm(
        points_au,
        axis=1,
    )

    nearest_cell_radius = np.min(radius_xy)

    zoom_radius = np.clip(
        ZOOM_RADIUS_FACTOR
        * nearest_cell_radius,
        ZOOM_RADIUS_MIN_AU,
        ZOOM_RADIUS_MAX_AU,
    )

    # ========================================================
    # ズーム領域
    # ========================================================
    zoom_mask = (
        radius_xy <= zoom_radius
    )

    points_zoom = points_au[zoom_mask]
    density_zoom_raw = density[zoom_mask]

    # ズーム領域内の球半径
    radius_spherical_zoom = (
        radius_spherical[zoom_mask]
    )

    # ========================================================
    # シンク外部だけを選ぶ診断用マスク
    # ========================================================
    outside_sink_mask = (
        radius_spherical_zoom
        >= SINK_MASK_RADIUS_AU
    )

    if not np.any(outside_sink_mask):
        print(
            f"[WARNING] No cells outside sink "
            f"at timestep {timestep:05d}"
        )
        continue

    # 最大・最小値とカラースケールの計算に使用
    points_diagnostic = points_zoom[
        outside_sink_mask
    ]

    density_diagnostic = density_zoom_raw[
        outside_sink_mask
    ]

    radius_diagnostic = radius_spherical_zoom[
        outside_sink_mask
    ]

    if len(points_zoom) <= 10:
        print(
            f"[WARNING] Insufficient zoom data "
            f"at timestep {timestep:05d}"
        )
        continue

    # 補間グリッド
    zoom_axis = np.linspace(
        -zoom_radius,
        zoom_radius,
        ZOOM_RESOLUTION,
    )

    X_zoom, Y_zoom = np.meshgrid(
        zoom_axis,
        zoom_axis,
    )

    density_zoom = interpolate_density_smooth(
        points_zoom[:, [0, 1]],
        density_zoom_raw,
        X_zoom,
        Y_zoom,
    )

    # シンク内部を除外した密度から
    # カラースケールを決定
    zoom_vmin, zoom_vmax = (
        calculate_log_limits(
            density_diagnostic,
            lower_percentile=(
                ZOOM_COLOR_PERCENTILES[0]
            ),
            upper_percentile=(
                ZOOM_COLOR_PERCENTILES[1]
            ),
        )
    )

    zoom_norm = LogNorm(
        vmin=zoom_vmin,
        vmax=zoom_vmax,
    )

    # ========================================================
    # 最大・最小密度
    # シンク外部だけを検索
    # ========================================================
    maximum_index = np.argmax(
        density_diagnostic
    )

    minimum_index = np.argmin(
        density_diagnostic
    )

    rho_max = density_diagnostic[
        maximum_index
    ]

    rho_min = density_diagnostic[
        minimum_index
    ]

    r_max = radius_diagnostic[
        maximum_index
    ]

    r_min = radius_diagnostic[
        minimum_index
    ]

    position_max = points_diagnostic[
        maximum_index
    ]

    position_min = points_diagnostic[
        minimum_index
    ]

    print(
        f"[INFO] Density extrema outside sink "
        f"(r >= {SINK_MASK_RADIUS_AU:.3e} AU)"
    )

    print(
        f"  rho_max = {rho_max:.3e} "
        f"M_sun AU^-3 at "
        f"({position_max[0]:.3e}, "
        f"{position_max[1]:.3e}, "
        f"{position_max[2]:.3e}) AU, "
        f"r = {r_max:.3e} AU"
    )

    print(
        f"  rho_min = {rho_min:.3e} "
        f"M_sun AU^-3 at "
        f"({position_min[0]:.3e}, "
        f"{position_min[1]:.3e}, "
        f"{position_min[2]:.3e}) AU, "
        f"r = {r_min:.3e} AU"
    )

    # ========================================================
    # Figure
    # ========================================================
    fig = plt.figure(figsize=figsize)

    gs = GridSpec(
        1,
        3,
        width_ratios=[5.0, 0.3, 1.8],
        wspace=0.35,
    )

    ax = fig.add_subplot(gs[0, 0])
    ax.set_aspect("equal")

    image = ax.imshow(
        density_zoom,
        origin="lower",
        extent=[
            -zoom_radius,
            zoom_radius,
            -zoom_radius,
            zoom_radius,
        ],
        cmap=ZOOM_CMAP,
        norm=zoom_norm,
        interpolation=ZOOM_IMAGE_INTERPOLATION,
        aspect="equal",
        rasterized=True,
    )

    ax.plot(
        0.0,
        0.0,
        "r+",
        markersize=12,
        markeredgewidth=2,
        label="Center",
    )

    ax.set_xlim(
        -zoom_radius,
        zoom_radius,
    )
    ax.set_ylim(
        -zoom_radius,
        zoom_radius,
    )

    ax.set_xlabel(
        r"$x\ [{\rm AU}]$",
        fontsize=12,
    )
    ax.set_ylabel(
        r"$y\ [{\rm AU}]$",
        fontsize=12,
    )

    ax.set_title(
        "Zoomed Density Map: x-y Midplane\n"
        f"t = {time_yr:.6e} yr",
        fontsize=13,
        fontweight="bold",
    )

    ax.ticklabel_format(
        axis="both",
        style="scientific",
        scilimits=(0, 0),
    )

    # 通常のグリッド線も非表示
    ax.grid(False)

    ax.legend(
        loc="upper right",
        fontsize=8,
        framealpha=0.7,
    )

    # カラーバー
    cax = fig.add_subplot(gs[0, 1])

    colorbar = fig.colorbar(
        image,
        cax=cax,
        extend="both",
    )

    colorbar.ax.yaxis.set_major_formatter(
        LogFormatterSciNotation()
    )

    colorbar.set_label(
        r"$\rho\ "
        r"[M_\odot\,{\rm AU}^{-3}]$"
        "\nZoom 1–99 percentile",
        fontsize=10,
    )

    # 情報欄
    ax_info = fig.add_subplot(gs[0, 2])
    ax_info.axis("off")

    information_text = (
    "Time\n"
    f"{time_yr:.6e} yr\n\n"

    "Code time\n"
    f"{time_code:.6e}\n\n"

    "Zoom radius\n"
    f"{zoom_radius:.3e} AU\n\n"

    "Density search region\n"
    f"r >= {SINK_MASK_RADIUS_AU:.3e} AU\n"
    "(outside sink)\n\n"

    "Maximum density\n"
    f"{rho_max:.3e}\n"
    r"$M_\odot\,{\rm AU}^{-3}$"
    "\n"
    f"r = {r_max:.3e} AU\n\n"

    "Minimum density\n"
    f"{rho_min:.3e}\n"
    r"$M_\odot\,{\rm AU}^{-3}$"
    "\n"
    f"r = {r_min:.3e} AU\n\n"

    "Color range\n"
    f"{zoom_vmin:.2e}\n"
    "to\n"
    f"{zoom_vmax:.2e}"
)

    ax_info.text(
        0.04,
        0.95,
        information_text,
        transform=ax_info.transAxes,
        verticalalignment="top",
        fontsize=10,
        bbox=dict(
            boxstyle="round",
            facecolor="whitesmoke",
            edgecolor="black",
            alpha=0.9,
        ),
    )

    plt.subplots_adjust(
        top=0.90,
        bottom=0.10,
        left=0.08,
        right=0.97,
    )

    # ========================================================
    # タイムステップ順のファイル名で保存
    # ========================================================
    png = os.path.join(
        output_dir,
        (
            f"density_xy_timestep_"
            f"{timestep:05d}_"
            f"time_{time_yr:.6e}yr_"
            "zoom_no_grid.png"
        ),
    )

    fig.savefig(
        png,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)

    print(
        f"[INFO] Saved: timestep="
        f"{timestep:05d}, "
        f"time={time_yr:.6e} yr"
    )


print(
    f"[INFO] All x-y zoom maps saved to: "
    f"{output_dir}"
)

# ============================================================
# x-z平面・ズーム密度マップのみ
#
# ・AMR境界線なし
# ・通常のグリッド線なし
# ・y=0に最も近いセル層を1層だけ使用
# ・距離：AU
# ・密度：M_sun AU^-3
# ・VTKヘッダーのコード時刻をyrへ変換
# ・線形補間＋最近傍補間による穴埋め
# ・bilinear表示
# ・ズーム領域内の1–99パーセンタイルで色範囲を調整
# ・ファイル名はタイムステップ順
# ============================================================

import os
import re
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import LogFormatterSciNotation
from scipy.interpolate import griddata


# ============================================================
# 入出力設定
# ============================================================
vtk_dir = os.path.expanduser(
    "~/athena-project/results/〇〇"
)

output_dir = "./xz_density_zoom_no_grid"
os.makedirs(output_dir, exist_ok=True)


# ============================================================
# 単位定義
# ============================================================
# Toyouchi.cppのコード単位
M_UNIT_CGS = 4.0e33   # g
L_UNIT_CGS = 6.7e15   # cm
T_UNIT_CGS = 3.34e10  # s

# 物理定数
AU_CGS = 1.495978707e13
MSUN_CGS = 1.98847e33
YEAR_CGS = 365.25 * 24.0 * 3600.0

# コード単位 → 表示単位
LENGTH_UNIT_AU = L_UNIT_CGS / AU_CGS
MASS_UNIT_MSUN = M_UNIT_CGS / MSUN_CGS
TIME_UNIT_YR = T_UNIT_CGS / YEAR_CGS

# code density → M_sun AU^-3
DENSITY_UNIT_MSUN_AU3 = (
    MASS_UNIT_MSUN / LENGTH_UNIT_AU**3
)

print("[INFO] Unit conversion factors:")
print(
    f"  1 code length  = "
    f"{LENGTH_UNIT_AU:.6e} AU"
)
print(
    f"  1 code mass    = "
    f"{MASS_UNIT_MSUN:.6e} M_sun"
)
print(
    f"  1 code time    = "
    f"{TIME_UNIT_YR:.6e} yr"
)
print(
    f"  1 code density = "
    f"{DENSITY_UNIT_MSUN_AU3:.6e} "
    "M_sun AU^-3"
)


# ============================================================
# ズーム設定
# ============================================================
ZOOM_RESOLUTION = 800
ZOOM_RADIUS_FACTOR = 20.0

# 計算領域用のズーム半径
# AU単位で直接指定する
ZOOM_RADIUS_MIN_AU = 2.0e4   # 最小ズーム半径：2,0000 AU
ZOOM_RADIUS_MAX_AU = 5.0e4   # 最大ズーム半径：50,000 AU



# ズーム領域内の色範囲
ZOOM_COLOR_PERCENTILES = (5.0, 98.0)

ZOOM_CMAP = "turbo"
ZOOM_IMAGE_INTERPOLATION = "bilinear"

dpi = 200
figsize = (10, 8)

# ============================================================
# シンク領域のマスク設定
# inputファイルのr_sink_auと一致させる
# ============================================================
SINK_RADIUS_AU = 1000.0

# シンク境界直近まで除外する場合は1.1～1.2に変更
SINK_MASK_FACTOR = 1.0

SINK_MASK_RADIUS_AU = (
    SINK_MASK_FACTOR * SINK_RADIUS_AU
)

# ============================================================
# VTKヘッダーから時刻を取得
# ============================================================
def read_vtk_time_code(filename):
    with open(filename, "rb") as vtk_file:
        header = vtk_file.read(512).decode(
            "ascii",
            errors="ignore",
        )

    match = re.search(
        r"time\s*=\s*"
        r"([+-]?(?:\d+\.?\d*|\.\d+)"
        r"(?:[eE][+-]?\d+)?)",
        header,
    )

    if match is None:
        raise ValueError(
            f"Could not find time in VTK header: "
            f"{filename}"
        )

    return float(match.group(1))


# ============================================================
# 対数カラースケール
# ============================================================
def calculate_log_limits(
    values,
    lower_percentile=1.0,
    upper_percentile=99.0,
):
    values = np.asarray(values).ravel()

    positive_values = values[
        np.isfinite(values)
        & (values > 0.0)
    ]

    if len(positive_values) == 0:
        raise ValueError(
            "No positive finite density values."
        )

    vmin = np.percentile(
        positive_values,
        lower_percentile,
    )
    vmax = np.percentile(
        positive_values,
        upper_percentile,
    )

    if not np.isfinite(vmin) or vmin <= 0.0:
        vmin = np.min(positive_values)

    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = np.max(positive_values)

    if vmax <= vmin:
        vmin *= 0.5
        vmax *= 2.0

    if vmax / vmin < 10.0:
        center = np.sqrt(vmin * vmax)
        vmin = center / np.sqrt(10.0)
        vmax = center * np.sqrt(10.0)

    return vmin, vmax


# ============================================================
# y=0に最も近いセル層を抽出
# ============================================================
def extract_xz_midplane(
    grid,
    density_name,
):
    """
    y=0と交差するブロックから、y=0に最も近いセル中心面を
    1層だけ抽出する。
    """
    bounds = grid.bounds

    # bounds = (xmin, xmax, ymin, ymax, zmin, zmax)
    if not (
        bounds[2] <= 0.0 <= bounds[3]
    ):
        return None, None

    points_code = grid.cell_centers().points

    if len(points_code) == 0:
        return None, None

    density_code = np.asarray(
        grid[density_name]
    ).reshape(-1)

    if len(density_code) != len(points_code):
        raise ValueError(
            "Density size does not match "
            "cell-center size."
        )

    y_values = points_code[:, 1]

    y_nearest = y_values[
        np.argmin(np.abs(y_values))
    ]

    tolerance = (
        1.0e-10
        * max(np.max(np.abs(y_values)), 1.0)
    )

    midplane_mask = np.isclose(
        y_values,
        y_nearest,
        rtol=1.0e-10,
        atol=tolerance,
    )

    if not np.any(midplane_mask):
        return None, None

    points_au = (
        points_code[midplane_mask]
        * LENGTH_UNIT_AU
    )

    density_msun_au3 = (
        density_code[midplane_mask]
        * DENSITY_UNIT_MSUN_AU3
    )

    return points_au, density_msun_au3


# ============================================================
# 線形補間＋最近傍補間
# ============================================================
def interpolate_density_smooth(
    points_2d,
    density,
    X,
    Z,
):
    density_linear = griddata(
        points_2d,
        density,
        (X, Z),
        method="linear",
    )

    density_nearest = griddata(
        points_2d,
        density,
        (X, Z),
        method="nearest",
    )

    density_combined = np.where(
        np.isfinite(density_linear),
        density_linear,
        density_nearest,
    )

    invalid = (
        ~np.isfinite(density_combined)
        | (density_combined <= 0.0)
    )
    density_combined[invalid] = np.nan

    return density_combined


# ============================================================
# VTKファイル整理
# ============================================================
if not os.path.isdir(vtk_dir):
    raise FileNotFoundError(
        f"VTK directory does not exist: {vtk_dir}"
    )

files_by_step = defaultdict(list)

for filename in os.listdir(vtk_dir):
    if not (
        filename.startswith("Toyouchi.block")
        and filename.endswith(".vtk")
    ):
        continue

    match = re.search(
        r"(?:prim\.)?out2\.(\d+)",
        filename,
    )

    if match is not None:
        timestep = int(match.group(1))

        files_by_step[timestep].append(
            os.path.join(
                vtk_dir,
                filename,
            )
        )

timesteps = sorted(files_by_step)

if not timesteps:
    raise RuntimeError(
        f"No VTK files found in {vtk_dir}"
    )

print(
    f"[INFO] Found {len(timesteps)} timesteps"
)
print(
    f"[INFO] Timestep range: "
    f"{timesteps[0]}–{timesteps[-1]}"
)


# ============================================================
# 密度変数名検出
# ============================================================
test_file = files_by_step[timesteps[0]][0]
test_grid = pv.read(test_file)

density_name = next(
    (
        name
        for name in [
            "dens",
            "density",
            "rho",
            "prim_dens",
            "prim_density",
        ]
        if name in test_grid.array_names
    ),
    None,
)

if density_name is None:
    raise RuntimeError(
        "No density array was found."
    )

print(
    f"[INFO] Density variable: {density_name}"
)


# ============================================================
# メイン処理
# ============================================================
for timestep_index, timestep in enumerate(
    timesteps
):
    print(
        f"[INFO] Processing timestep "
        f"{timestep:05d} "
        f"({timestep_index + 1}/"
        f"{len(timesteps)})"
    )

    points_list = []
    density_list = []

    for filename in files_by_step[timestep]:
        try:
            grid = pv.read(filename)

            points_au, density = (
                extract_xz_midplane(
                    grid,
                    density_name,
                )
            )

            if points_au is not None:
                points_list.append(points_au)
                density_list.append(density)

        except Exception as error:
            print(
                f"[WARNING] {filename}: {error}"
            )

    if not points_list:
        print(
            f"[WARNING] No x-z midplane data "
            f"at timestep {timestep:05d}"
        )
        continue

    points_au = np.vstack(points_list)
    density = np.hstack(density_list)

    valid = (
        np.all(
            np.isfinite(points_au),
            axis=1,
        )
        & np.isfinite(density)
        & (density > 0.0)
    )

    points_au = points_au[valid]
    density = density[valid]

    if len(density) == 0:
        continue

    # 物理時刻
    representative_file = (
        files_by_step[timestep][0]
    )

    time_code = read_vtk_time_code(
        representative_file
    )
    time_yr = time_code * TIME_UNIT_YR

    # x-z平面内半径
    radius_xz = np.hypot(
        points_au[:, 0],
        points_au[:, 2],
    )
    
    # シンク判定用の球半径
    # シミュレーション本体のr_sphと同じ定義
    radius_spherical = np.linalg.norm(
        points_au,
        axis=1,
    )

    nearest_cell_radius = np.min(radius_xz)

    zoom_radius = np.clip(
        ZOOM_RADIUS_FACTOR
        * nearest_cell_radius,
        ZOOM_RADIUS_MIN_AU,
        ZOOM_RADIUS_MAX_AU,
    )

    # ========================================================
    # ズーム領域
    # ========================================================
    zoom_mask = (
        radius_xz <= zoom_radius
    )

    points_zoom = points_au[zoom_mask]
    density_zoom_raw = density[zoom_mask]

    # ズーム領域内の球半径
    radius_spherical_zoom = (
        radius_spherical[zoom_mask]
    )

    if len(points_zoom) <= 10:
        print(
            f"[WARNING] Insufficient zoom data "
            f"at timestep {timestep:05d}"
        )
        continue

    # ========================================================
    # シンク外部のみを選択する診断用マスク
    # ========================================================
    outside_sink_mask = (
        radius_spherical_zoom
        >= SINK_MASK_RADIUS_AU
    )

    if not np.any(outside_sink_mask):
        print(
            f"[WARNING] No cells outside sink "
            f"at timestep {timestep:05d}"
        )
        continue

    # 最大・最小値およびカラースケール計算用
    points_diagnostic = points_zoom[
        outside_sink_mask
    ]

    density_diagnostic = density_zoom_raw[
        outside_sink_mask
    ]

    radius_diagnostic = radius_spherical_zoom[
        outside_sink_mask
    ]

    if len(points_zoom) <= 10:
        print(
            f"[WARNING] Insufficient zoom data "
            f"at timestep {timestep:05d}"
        )
        continue

    # 補間グリッド
    zoom_axis = np.linspace(
        -zoom_radius,
        zoom_radius,
        ZOOM_RESOLUTION,
    )

    X_zoom, Z_zoom = np.meshgrid(
        zoom_axis,
        zoom_axis,
    )

    # xとzを補間座標として使用
    density_zoom = interpolate_density_smooth(
        points_zoom[:, [0, 2]],
        density_zoom_raw,
        X_zoom,
        Z_zoom,
    )

    # ========================================================
    # ズーム専用カラースケール
    # シンク内部を除外した密度を使用
    # ========================================================
    zoom_vmin, zoom_vmax = (
        calculate_log_limits(
            density_diagnostic,
            lower_percentile=(
                ZOOM_COLOR_PERCENTILES[0]
            ),
            upper_percentile=(
                ZOOM_COLOR_PERCENTILES[1]
            ),
        )
    )

    zoom_norm = LogNorm(
        vmin=zoom_vmin,
        vmax=zoom_vmax,
    )

    zoom_norm = LogNorm(
        vmin=zoom_vmin,
        vmax=zoom_vmax,
    )

    # ========================================================
    # 最大・最小密度
    # シンク外かつズーム領域内のみ検索
    # ========================================================
    maximum_index = np.argmax(
        density_diagnostic
    )

    minimum_index = np.argmin(
        density_diagnostic
    )

    rho_max = density_diagnostic[
        maximum_index
    ]

    rho_min = density_diagnostic[
        minimum_index
    ]

    r_max = radius_diagnostic[
        maximum_index
    ]

    r_min = radius_diagnostic[
        minimum_index
    ]

    position_max = points_diagnostic[
        maximum_index
    ]

    position_min = points_diagnostic[
        minimum_index
    ]

    print(
        f"[INFO] Density extrema outside sink "
        f"(r >= {SINK_MASK_RADIUS_AU:.3e} AU)"
    )

    print(
        f"  rho_max = {rho_max:.3e} "
        f"M_sun AU^-3 at "
        f"(x, y, z) = "
        f"({position_max[0]:.3e}, "
        f"{position_max[1]:.3e}, "
        f"{position_max[2]:.3e}) AU, "
        f"r = {r_max:.3e} AU"
    )

    print(
        f"  rho_min = {rho_min:.3e} "
        f"M_sun AU^-3 at "
        f"(x, y, z) = "
        f"({position_min[0]:.3e}, "
        f"{position_min[1]:.3e}, "
        f"{position_min[2]:.3e}) AU, "
        f"r = {r_min:.3e} AU"
    )

    # ========================================================
    # Figure
    # ========================================================
    fig = plt.figure(figsize=figsize)

    gs = GridSpec(
        1,
        3,
        width_ratios=[5.0, 0.3, 1.8],
        wspace=0.35,
    )

    ax = fig.add_subplot(gs[0, 0])
    ax.set_aspect("equal")

    image = ax.imshow(
        density_zoom,
        origin="lower",
        extent=[
            -zoom_radius,
            zoom_radius,
            -zoom_radius,
            zoom_radius,
        ],
        cmap=ZOOM_CMAP,
        norm=zoom_norm,
        interpolation=ZOOM_IMAGE_INTERPOLATION,
        aspect="equal",
        rasterized=True,
    )

    ax.plot(
        0.0,
        0.0,
        "r+",
        markersize=12,
        markeredgewidth=2,
        label="Center",
    )

    ax.set_xlim(
        -zoom_radius,
        zoom_radius,
    )
    ax.set_ylim(
        -zoom_radius,
        zoom_radius,
    )

    ax.set_xlabel(
        r"$x\ [{\rm AU}]$",
        fontsize=12,
    )
    ax.set_ylabel(
        r"$z\ [{\rm AU}]$",
        fontsize=12,
    )

    ax.set_title(
        "Zoomed Density Map: x-z Midplane\n"
        f"t = {time_yr:.6e} yr",
        fontsize=13,
        fontweight="bold",
    )

    ax.ticklabel_format(
        axis="both",
        style="scientific",
        scilimits=(0, 0),
    )

    # 通常のグリッド線も非表示
    ax.grid(False)

    ax.legend(
        loc="upper right",
        fontsize=8,
        framealpha=0.7,
    )

    # カラーバー
    cax = fig.add_subplot(gs[0, 1])

    colorbar = fig.colorbar(
        image,
        cax=cax,
        extend="both",
    )

    colorbar.ax.yaxis.set_major_formatter(
        LogFormatterSciNotation()
    )

    colorbar.set_label(
        r"$\rho\ "
        r"[M_\odot\,{\rm AU}^{-3}]$"
        "\nZoom 1–99 percentile",
        fontsize=10,
    )

    # 情報欄
    ax_info = fig.add_subplot(gs[0, 2])
    ax_info.axis("off")

    
    information_text = (
    "Time\n"
    f"{time_yr:.6e} yr\n\n"

    "Code time\n"
    f"{time_code:.6e}\n\n"

    "Zoom radius\n"
    f"{zoom_radius:.3e} AU\n\n"

    "Density search region\n"
    f"r >= {SINK_MASK_RADIUS_AU:.3e} AU\n"
    "(outside sink)\n\n"

    "Maximum density\n"
    f"{rho_max:.3e}\n"
    r"$M_\odot\,{\rm AU}^{-3}$"
    "\n"
    f"r = {r_max:.3e} AU\n\n"

    "Minimum density\n"
    f"{rho_min:.3e}\n"
    r"$M_\odot\,{\rm AU}^{-3}$"
    "\n"
    f"r = {r_min:.3e} AU\n\n"

    "Color range\n"
    f"{zoom_vmin:.2e}\n"
    "to\n"
    f"{zoom_vmax:.2e}"
)
    

    ax_info.text(
        0.04,
        0.95,
        information_text,
        transform=ax_info.transAxes,
        verticalalignment="top",
        fontsize=10,
        bbox=dict(
            boxstyle="round",
            facecolor="whitesmoke",
            edgecolor="black",
            alpha=0.9,
        ),
    )

    plt.subplots_adjust(
        top=0.90,
        bottom=0.10,
        left=0.08,
        right=0.97,
    )

    # ========================================================
    # タイムステップ順のファイル名で保存
    # ========================================================
    png = os.path.join(
        output_dir,
        (
            f"density_xz_timestep_"
            f"{timestep:05d}_"
            f"time_{time_yr:.6e}yr_"
            "zoom_no_grid.png"
        ),
    )

    fig.savefig(
        png,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)

    print(
        f"[INFO] Saved: timestep="
        f"{timestep:05d}, "
        f"time={time_yr:.6e} yr"
    )


print(
    f"[INFO] All x-z zoom maps saved to: "
    f"{output_dir}"
)